# Spin BuAli — STT benchmark (Kaggle, dual T4)

Runs a batch of radiology dictations through several Persian STT models and
ranks them on accuracy, clinical correctness, speed and memory.

**Self-contained.** No repo to clone, no dataset to attach. Set the accelerator
to **GPU T4 ×2** and Internet to **On** (the model weights come from Hugging
Face), then Run All.

**Before you edit a cell in section 2:** those cells are generated from the
project's source files and are covered by its test suite. Change the repo and
re-run `benchmark/notebooks/build_notebook.py`, rather than patching here — an
edit made in the notebook is lost on the next regeneration.

## 1. Check the hardware

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,memory.used --format=csv

In [ ]:
import torch

print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()}  "
      f"devices={torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i}  {p.name}  {p.total_memory / 1024**3:.1f} GB")

assert torch.cuda.device_count() >= 1, "no GPU — set Accelerator to GPU T4 x2"
if torch.cuda.device_count() == 1:
    print("\nonly one GPU visible: the run will work but take about twice as long")

In [ ]:
# Kaggle already ships torch, transformers, soundfile and pandas. This adds only
# what the benchmark needs on top, and stays quiet when they are present.
!pip install -q python-dotenv sentencepiece

## 2. The code

The next cells write the benchmark's modules to disk and put them on the import
path. They are the project's real source files, embedded — read them, but change
them in the repo.

In [ ]:
import pathlib, sys

SRC = pathlib.Path("/kaggle/working/buali_src")
if not pathlib.Path("/kaggle/working").is_dir():
    SRC = pathlib.Path.cwd() / "buali_src"   # so the notebook also runs locally


def write(relative_path, text):
    path = SRC / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    return path


# stt/ is a package; the other two are plain directories of modules.
(SRC / "stt" / "app").mkdir(parents=True, exist_ok=True)
(SRC / "stt" / "app" / "__init__.py").write_text("", encoding="utf-8")

# Only benchmark/ goes on the path here. It reaches the other two itself, the
# same way it does in the repo — one place decides, and it is a source file.
sys.path.insert(0, str(SRC / "benchmark"))
print("source tree:", SRC)

### The metrics

Straight from `evaluation/` — the same functions the live scoring service runs.
Not a reimplementation: if a metric changes there, this notebook changes with
it, so the benchmark can never quietly disagree with production about which
model is better.

In [ ]:
write("evaluation/config.py", r'''
"""Evaluation service configuration, from the environment or `evaluation/.env`."""
import os
import pathlib

from dotenv import load_dotenv

load_dotenv(pathlib.Path(__file__).parent / ".env")

HOST = os.getenv("HOST", "0.0.0.0")
PORT = int(os.getenv("PORT", "8002"))

# The concept vocabulary. Point this elsewhere to score against a different
# term set without touching the code.
CLINICAL_TERMS_PATH = os.getenv(
    "CLINICAL_TERMS_PATH",
    str(pathlib.Path(__file__).parent / "clinical_terms.json"),
)

# --- Optional embedding metrics (semantic_metrics.py) ---
# Only loaded when a request asks for them. See requirements-semantic.txt.
BERTSCORE_MODEL = os.getenv("BERTSCORE_MODEL", "HooshvareLab/bert-base-parsbert-uncased")
BERTSCORE_LAYERS = int(os.getenv("BERTSCORE_LAYERS", "8"))
SIMILARITY_MODEL = os.getenv(
    "SIMILARITY_MODEL", "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# A report worse than this counts as catastrophic in the batch summary.
CATASTROPHIC_WER = float(os.getenv("CATASTROPHIC_WER", "1.0"))

# --- Degenerate-output thresholds ---
# A looping or wildly padded output is unusable regardless of how the clinical
# counters read, so these also raise requires_medical_review.
MAX_REPETITION = float(os.getenv("MAX_REPETITION", "0.5"))
MAX_LENGTH_RATIO = float(os.getenv("MAX_LENGTH_RATIO", "2.0"))
MIN_LENGTH_RATIO = float(os.getenv("MIN_LENGTH_RATIO", "0.5"))
''')

In [ ]:
write("evaluation/text_normalizer.py", r'''
"""Normalisation — the step every metric depends on.

Both sides of a comparison go through `normalize()` before anything is counted,
so ۱۰۷ and 107, "می‌رود" and "میرود", and "شش" and "6" are not scored as
differences. Nothing downstream should have to think about script or spacing.

The rules are deliberately conservative: they unify representations of the
*same* token and never rewrite clinical content.
"""
import re
import unicodedata

# --- Character tables ------------------------------------------------------
PERSIAN_DIGITS = "۰۱۲۳۴۵۶۷۸۹"      # U+06F0..U+06F9
ARABIC_DIGITS = "٠١٢٣٤٥٦٧٨٩"       # U+0660..U+0669
_DIGIT_MAP = {ord(c): str(i) for i, c in enumerate(PERSIAN_DIGITS)}
_DIGIT_MAP.update({ord(c): str(i) for i, c in enumerate(ARABIC_DIGITS)})

# Arabic letter forms that Persian keyboards and STT engines emit
# interchangeably with their Persian counterparts.
_LETTER_MAP = {
    "ي": "ی", "ك": "ک", "ة": "ه", "ۀ": "ه",
    "أ": "ا", "إ": "ا", "ٱ": "ا",
    "ؤ": "و", "ئ": "ی",
}

# Harakat/tanwin, superscript alef, and kashida carry no lexical weight here.
_DIACRITICS = re.compile("[ً-ْٰـ]")

# Zero-width and bidi controls. ZWNJ (U+200C) is handled separately below.
_INVISIBLE = re.compile("[‍‎‏‪-‮⁦-⁩﻿]")
ZWNJ = "‌"

_PUNCTUATION = re.compile(r"[^\w\s.]", re.UNICODE)
# A dot only survives when it sits between two digits (a decimal point).
_STRAY_DOT = re.compile(r"(?<!\d)\.|\.(?!\d)")
_WHITESPACE = re.compile(r"\s+")

# --- Spelled-out numbers ---------------------------------------------------
_PERSIAN_UNITS = {
    "صفر": 0, "یک": 1, "دو": 2, "سه": 3, "چهار": 4, "پنج": 5, "شش": 6, "شیش": 6,
    "هفت": 7, "هشت": 8, "نه": 9, "ده": 10, "یازده": 11, "دوازده": 12,
    "سیزده": 13, "چهارده": 14, "پانزده": 15, "پونزده": 15, "شانزده": 16,
    "شونزده": 16, "هفده": 17, "هیفده": 17, "هجده": 18, "هیجده": 18, "نوزده": 19,
}
_PERSIAN_TENS = {
    "بیست": 20, "سی": 30, "چهل": 40, "پنجاه": 50, "شصت": 60,
    "هفتاد": 70, "هشتاد": 80, "نود": 90,
}
_PERSIAN_HUNDREDS = {
    "صد": 100, "یکصد": 100, "دویست": 200, "سیصد": 300, "چهارصد": 400,
    "پانصد": 500, "پونصد": 500, "ششصد": 600, "هفتصد": 700,
    "هشتصد": 800, "نهصد": 900,
}
_PERSIAN_SCALES = {"هزار": 1000, "میلیون": 1000000}

_ENGLISH_UNITS = {
    "zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6,
    "seven": 7, "eight": 8, "nine": 9, "ten": 10, "eleven": 11, "twelve": 12,
    "thirteen": 13, "fourteen": 14, "fifteen": 15, "sixteen": 16,
    "seventeen": 17, "eighteen": 18, "nineteen": 19,
}
_ENGLISH_TENS = {
    "twenty": 20, "thirty": 30, "forty": 40, "fourty": 40, "fifty": 50,
    "sixty": 60, "seventy": 70, "eighty": 80, "ninety": 90,
}
_ENGLISH_SCALES = {"hundred": 100, "thousand": 1000, "million": 1000000}

# "و" joins number words in Persian ("بیست و دو"), but is also an ordinary
# conjunction, so it only counts as a joiner between two number words.
_PERSIAN_JOINER = "و"
_NUMBER_WORDS = (
    set(_PERSIAN_UNITS) | set(_PERSIAN_TENS) | set(_PERSIAN_HUNDREDS)
    | set(_PERSIAN_SCALES) | set(_ENGLISH_UNITS) | set(_ENGLISH_TENS)
    | set(_ENGLISH_SCALES)
)


def _word_value(word):
    for table in (_PERSIAN_UNITS, _PERSIAN_TENS, _PERSIAN_HUNDREDS,
                  _ENGLISH_UNITS, _ENGLISH_TENS):
        if word in table:
            return table[word]
    return None


def _scale_value(word):
    return _PERSIAN_SCALES.get(word) or _ENGLISH_SCALES.get(word)


def _collapse_number_run(words):
    """Fold a run of number words into one integer, or None if it isn't one.

    Handles additive forms ("بیست و دو", "twenty two") and multiplicative
    scales ("three hundred", "دو هزار").
    """
    total = 0
    current = 0
    seen = False
    for word in words:
        if word == _PERSIAN_JOINER:
            continue
        scale = _scale_value(word)
        if scale is not None:
            current = (current or 1) * scale
            if scale >= 1000:
                total += current
                current = 0
            seen = True
            continue
        value = _word_value(word)
        if value is None:
            return None
        current += value
        seen = True
    return (total + current) if seen else None


def words_to_numbers(text):
    """Replace runs of spelled-out numbers with digits, so that
    "شش میلی‌متر" and "6 mm" agree once both sides are normalised."""
    tokens = text.split()
    out = []
    index = 0
    while index < len(tokens):
        if tokens[index] not in _NUMBER_WORDS:
            out.append(tokens[index])
            index += 1
            continue
        # Extend while the run still looks numeric. A trailing joiner is left
        # alone rather than swallowing the following clause.
        end = index
        while end < len(tokens) and (
            tokens[end] in _NUMBER_WORDS
            or (tokens[end] == _PERSIAN_JOINER
                and end + 1 < len(tokens) and tokens[end + 1] in _NUMBER_WORDS)
        ):
            end += 1
        run = tokens[index:end]
        value = _collapse_number_run(run)
        out.extend(run) if value is None else out.append(str(value))
        index = end
    return " ".join(out)


# --- Main entry point ------------------------------------------------------
def normalize(text, spell_out_numbers=True, strip_punctuation=True):
    """Canonical form of `text` for comparison.

    `spell_out_numbers` turns number words into digits; disable it for a
    verbatim view. `strip_punctuation` keeps decimal points inside numbers and
    drops everything else.
    """
    if not text:
        return ""

    text = unicodedata.normalize("NFC", text)
    text = _INVISIBLE.sub("", text)
    text = text.translate(_DIGIT_MAP)
    for source, target in _LETTER_MAP.items():
        text = text.replace(source, target)
    text = _DIACRITICS.sub("", text)

    # ZWNJ joins: "می‌رود" -> "میرود". The spaced variant stays two tokens,
    # which is the honest reading of what was actually written.
    text = text.replace(ZWNJ, "")

    text = text.replace("٫", ".").replace("٬", "").replace("،", " ")
    text = text.replace("×", " x ").replace("−", "-")
    text = text.lower()

    if strip_punctuation:
        # Strip punctuation, then drop only the dots outside a number.
        text = _STRAY_DOT.sub(" ", _PUNCTUATION.sub(" ", text))

    text = _WHITESPACE.sub(" ", text).strip()

    if spell_out_numbers:
        text = words_to_numbers(text)
    return text


def tokenize(text, **kwargs):
    """Normalised whitespace tokens — the unit WER is measured in."""
    normalized = normalize(text, **kwargs)
    return normalized.split() if normalized else []
''')

In [ ]:
write("evaluation/extractors.py", r'''
"""Pulling structured facts out of normalised report text.

Everything here runs on `text_normalizer.normalize()` output, so patterns are
written in normalised form: ASCII digits, no ZWNJ, lowercase, no punctuation
except decimal points.

Four things get extracted, and each one backs a metric:
  measurements  -> number / unit error rates
  concepts      -> medical term precision / recall
  negation      -> negation error rate
  laterality    -> laterality error rate
"""
import hashlib
import json
import pathlib
import re
from dataclasses import dataclass

from text_normalizer import normalize

# --- Units -----------------------------------------------------------------
# Canonical unit per dimension: millimetres for length, millilitres for volume.
# Keys are post-normalisation spellings (ZWNJ already removed).
LENGTH_UNITS = {
    "mm": 1.0, "millimeter": 1.0, "millimeters": 1.0, "millimetre": 1.0,
    "میلیمتر": 1.0,
    "cm": 10.0, "centimeter": 10.0, "centimeters": 10.0, "centimetre": 10.0,
    "سانتیمتر": 10.0,
    "m": 1000.0, "meter": 1000.0, "متر": 1000.0,
}
VOLUME_UNITS = {
    "cc": 1.0, "ml": 1.0, "milliliter": 1.0, "milliliters": 1.0,
    "سیسی": 1.0, "میلیلیتر": 1.0,
}
_UNIT_DIMENSION = {unit: "length" for unit in LENGTH_UNITS}
_UNIT_DIMENSION.update({unit: "volume" for unit in VOLUME_UNITS})
_UNIT_FACTOR = dict(LENGTH_UNITS)
_UNIT_FACTOR.update(VOLUME_UNITS)

# Words joining the parts of a dimension group, e.g. 107 x 44 / 107 در 44.
_DIMENSION_JOINERS = {"x", "در", "by"}

_NUMBER = re.compile(r"^\d+(?:\.\d+)?$")


@dataclass(frozen=True)
class Measurement:
    """One numeric value with its unit, in canonical form."""
    value: float
    unit: str | None
    canonical: float
    dimension: str | None
    index: int

    def same_quantity(self, other):
        """True when both describe the same physical quantity."""
        if self.dimension != other.dimension:
            return False
        return abs(self.canonical - other.canonical) < 1e-9


def extract_measurements(text):
    """Numbers that are actually measurements.

    A bare number is ignored: in Persian the word for one doubles as the
    indefinite article, so counting every digit would invent measurements.
    A number qualifies when it carries a unit, or belongs to a dimension
    group such as 107 x 44, where a unit stated once applies to the group.
    """
    tokens = normalize(text).split()
    groups = []
    current = []

    for position, token in enumerate(tokens):
        if _NUMBER.match(token):
            current.append(position)
            continue
        if token in _DIMENSION_JOINERS and current:
            continue
        if current:
            groups.append(current)
            current = []
    if current:
        groups.append(current)

    measurements = []
    for group in groups:
        after = group[-1] + 1
        while after < len(tokens) and tokens[after] in _DIMENSION_JOINERS:
            after += 1
        unit = None
        if after < len(tokens) and tokens[after] in _UNIT_FACTOR:
            unit = tokens[after]

        if unit is None and len(group) < 2:
            continue

        for position in group:
            value = float(tokens[position])
            dimension = _UNIT_DIMENSION.get(unit) if unit else None
            factor = _UNIT_FACTOR.get(unit, 1.0) if unit else 1.0
            measurements.append(Measurement(
                value=value, unit=unit, canonical=value * factor,
                dimension=dimension, index=position,
            ))
    return measurements


# --- Negation --------------------------------------------------------------
# English negation precedes the finding (no stone); Persian negation usually
# follows it. Each cue therefore declares the direction it governs, and the
# window is searched that way.
NEGATION_CUES_BEFORE = {
    "no", "not", "without", "absent", "negative", "free", "denies",
    "unremarkable", "neither", "nor",
}
NEGATION_CUES_AFTER = {
    "ندارد", "نداره", "نداشت", "نیست", "نبود", "نشد", "نمیشود", "نمیشد",
    "ندارند", "nadarad", "nist", "nashod",
}
NEGATION_CUES_EITHER = {"بدون", "عدم", "فاقد", "منفی"}

# Tokens that close the scope, so "no stone but mass is seen" leaves the mass
# affirmed rather than negated.
_SCOPE_BREAKERS = {"but", "however", "اما", "ولی", "although", "though"}

NEGATION_WINDOW = 6


def negation_spans(tokens):
    """Token ranges (start, end) governed by a negation cue."""
    spans = []
    for position, token in enumerate(tokens):
        forward = token in NEGATION_CUES_BEFORE or token in NEGATION_CUES_EITHER
        backward = token in NEGATION_CUES_AFTER or token in NEGATION_CUES_EITHER
        if not (forward or backward):
            continue
        start = end = position
        if forward:
            while end + 1 < len(tokens) and end - position < NEGATION_WINDOW:
                if tokens[end + 1] in _SCOPE_BREAKERS:
                    break
                end += 1
        if backward:
            while start - 1 >= 0 and position - start < NEGATION_WINDOW:
                if tokens[start - 1] in _SCOPE_BREAKERS:
                    break
                start -= 1
        spans.append((start, end))
    return spans


def is_negated(tokens, index, spans=None):
    """Whether the token at `index` sits inside any negation scope."""
    spans = negation_spans(tokens) if spans is None else spans
    return any(start <= index <= end for start, end in spans)


# --- Laterality ------------------------------------------------------------
LATERALITY_TERMS = {
    "right": "right", "rt": "right", "راست": "right",
    "left": "left", "lt": "left", "چپ": "left",
    "bilateral": "bilateral", "both": "bilateral", "دوطرفه": "bilateral",
    "طرفین": "bilateral",
}
LATERALITY_WINDOW = 5


def laterality_at(tokens, index):
    """The laterality governing the token at `index`, if any.

    Uses the nearest marker within the window on either side, because word
    order differs between the two languages: right kidney / کلیه راست.
    """
    best = None
    best_distance = LATERALITY_WINDOW + 1
    low = max(0, index - LATERALITY_WINDOW)
    high = min(len(tokens), index + LATERALITY_WINDOW + 1)
    for position in range(low, high):
        side = LATERALITY_TERMS.get(tokens[position])
        if side is None:
            continue
        distance = abs(position - index)
        if distance < best_distance:
            best, best_distance = side, distance
    return best


# --- Clinical concepts -----------------------------------------------------
@dataclass(frozen=True)
class ConceptMention:
    concept_id: str
    category: str
    index: int
    negated: bool
    laterality: str | None


class ClinicalTerms:
    """The concept vocabulary, loaded from clinical_terms.json."""

    def __init__(self, path=None):
        path = pathlib.Path(path or pathlib.Path(__file__).parent / "clinical_terms.json")
        data = json.loads(path.read_text(encoding="utf-8"))
        self.version = data.get("version", "unknown")
        self.sha = hashlib.sha256(path.read_bytes()).hexdigest()[:8]
        self._by_phrase = {}
        for concept in data["concepts"]:
            for variant in concept["variants"]:
                phrase = tuple(normalize(variant).split())
                if phrase:
                    self._by_phrase[phrase] = (concept["id"], concept["category"])
        self.max_phrase_len = max((len(phrase) for phrase in self._by_phrase), default=1)

    def find(self, tokens):
        """Every concept mention in `tokens`.

        Longest phrase wins, so "fatty liver" is not also counted as "liver".
        """
        spans = negation_spans(tokens)
        mentions = []
        position = 0
        while position < len(tokens):
            for length in range(min(self.max_phrase_len, len(tokens) - position), 0, -1):
                hit = self._by_phrase.get(tuple(tokens[position:position + length]))
                if hit is None:
                    continue
                concept_id, category = hit
                mentions.append(ConceptMention(
                    concept_id=concept_id, category=category, index=position,
                    negated=is_negated(tokens, position, spans),
                    laterality=laterality_at(tokens, position),
                ))
                position += length
                break
            else:
                position += 1
        return mentions
''')

In [ ]:
write("evaluation/general_metrics.py", r'''
"""Text-level metrics: how much the two texts differ, regardless of meaning.

These say nothing clinical. They are here because they catch failure shapes the
concept-based metrics cannot see -- most importantly a model that collapses into
repeating itself, or one that returns far more text than was ever dictated.
"""
from collections import Counter
from dataclasses import dataclass

from text_normalizer import normalize, tokenize

# A hypothesis this much longer than its reference is padded, not transcribed.
REPETITION_ORDER = 4
CHRF_MAX_ORDER = 6
CHRF_BETA = 2.0  # recall weighted over precision, the sacreBLEU default

# Punctuation compared by Punctuation F1, in both scripts.
_PUNCTUATION = set(".,;:?!()-\"'«»،؛؟")

# Every Unicode block Persian is written in: Arabic, Arabic Supplement, Arabic
# Extended-A and -B, and the two presentation-form blocks.
_ARABIC_RANGES = [
    ("؀", "ۿ"), ("ݐ", "ݿ"), ("ࡰ", "࢟"),
    ("ࢠ", "ࣿ"), ("ﭐ", "﷿"), ("ﹰ", "﻿"),
]


# --- Edit distance ---------------------------------------------------------
@dataclass(frozen=True)
class EditCounts:
    substitutions: int
    insertions: int
    deletions: int
    reference_length: int

    @property
    def errors(self):
        return self.substitutions + self.insertions + self.deletions

    @property
    def rate(self):
        return self.errors / self.reference_length if self.reference_length else 0.0


def edit_counts(reference, hypothesis):
    """Levenshtein alignment, keeping the operation breakdown.

    Insertions are tracked separately because in a medical transcript they are
    the fabrication signal, not just noise.
    """
    rows, columns = len(reference), len(hypothesis)
    distance = [[0] * (columns + 1) for _ in range(rows + 1)]
    for row in range(rows + 1):
        distance[row][0] = row
    for column in range(columns + 1):
        distance[0][column] = column
    for row in range(1, rows + 1):
        for column in range(1, columns + 1):
            if reference[row - 1] == hypothesis[column - 1]:
                distance[row][column] = distance[row - 1][column - 1]
            else:
                distance[row][column] = 1 + min(
                    distance[row - 1][column - 1],  # substitution
                    distance[row][column - 1],      # insertion
                    distance[row - 1][column],      # deletion
                )

    substitutions = insertions = deletions = 0
    row, column = rows, columns
    while row > 0 or column > 0:
        if row > 0 and column > 0 and reference[row - 1] == hypothesis[column - 1] \
                and distance[row][column] == distance[row - 1][column - 1]:
            row, column = row - 1, column - 1
        elif row > 0 and column > 0 and distance[row][column] == distance[row - 1][column - 1] + 1:
            substitutions += 1
            row, column = row - 1, column - 1
        elif column > 0 and distance[row][column] == distance[row][column - 1] + 1:
            insertions += 1
            column -= 1
        else:
            deletions += 1
            row -= 1
    return EditCounts(substitutions, insertions, deletions, rows)


def word_error_rate(reference_text, hypothesis_text):
    return edit_counts(tokenize(reference_text), tokenize(hypothesis_text))


def character_error_rate(reference_text, hypothesis_text):
    return edit_counts(_characters(reference_text), _characters(hypothesis_text))


def _characters(text):
    return list(normalize(text).replace(" ", ""))


# --- chrF ------------------------------------------------------------------
def chrf(reference_text, hypothesis_text, max_order=CHRF_MAX_ORDER, beta=CHRF_BETA):
    """Character n-gram F-score.

    Gives partial credit where WER does not: a morphological variant shares
    most of its characters, which matters for Persian's attached suffixes.
    """
    reference = _characters(reference_text)
    hypothesis = _characters(hypothesis_text)
    if not reference and not hypothesis:
        return 1.0
    if not reference or not hypothesis:
        return 0.0

    precisions, recalls = [], []
    for order in range(1, max_order + 1):
        reference_grams = _ngrams(reference, order)
        hypothesis_grams = _ngrams(hypothesis, order)
        overlap = sum((reference_grams & hypothesis_grams).values())
        precisions.append(_ratio(overlap, sum(hypothesis_grams.values())))
        recalls.append(_ratio(overlap, sum(reference_grams.values())))

    precision = sum(precisions) / len(precisions)
    recall = sum(recalls) / len(recalls)
    if precision + recall == 0:
        return 0.0
    return (1 + beta ** 2) * precision * recall / (beta ** 2 * precision + recall)


def _ngrams(items, order):
    return Counter(tuple(items[i:i + order]) for i in range(len(items) - order + 1))


# --- Failure-shape signals -------------------------------------------------
def hallucination_ratio(reference_text, hypothesis_text):
    """Output length over reference length. Above 1 means extra text appeared.

    A blunt instrument, but it sees fabrication built from words that are not
    in the clinical vocabulary -- which the concept-based metrics cannot.
    """
    reference = len(tokenize(reference_text))
    hypothesis = len(tokenize(hypothesis_text))
    return _ratio(hypothesis, reference)


def repetition_score(hypothesis_text, order=REPETITION_ORDER):
    """Share of repeated n-grams in the output, from 0 (none) to near 1.

    This is the loop detector. A model that degenerates into emitting the same
    sentence over and over scores near 1 here while every clinical metric stays
    quiet, because the repeated content is often individually plausible.
    """
    tokens = tokenize(hypothesis_text)
    if len(tokens) < order:
        return 0.0
    grams = [tuple(tokens[i:i + order]) for i in range(len(tokens) - order + 1)]
    return 1.0 - _ratio(len(set(grams)), len(grams))


def punctuation_f1(reference_text, hypothesis_text):
    """Agreement on punctuation, compared as a multiset.

    Computed on the raw text, since normalisation deliberately strips
    punctuation before any other metric runs. Models that punctuate (Whisper)
    and models that emit bare text (Wav2Vec2, MMS) are otherwise not
    comparable on it at all.
    """
    reference = _punctuation(reference_text)
    hypothesis = _punctuation(hypothesis_text)
    if not reference and not hypothesis:
        return 1.0  # neither punctuates: perfect agreement, not a failure
    matched = sum((reference & hypothesis).values())
    precision = _ratio(matched, sum(hypothesis.values()))
    recall = _ratio(matched, sum(reference.values()))
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def _punctuation(text):
    return Counter(character for character in (text or "") if character in _PUNCTUATION)


def script_contamination(text):
    """Share of letters that are not Arabic-script.

    Computed on raw text, like punctuation F1, since normalisation does not
    remove Latin characters anyway.

    **Read this one against the reference, never on its own.** It was designed
    for corpora where any Latin character is leakage, and this one is not:
    these radiologists dictate English terms on purpose, so a perfectly correct
    transcript scores as heavily contaminated. What means something is the
    hypothesis figure compared with the reference figure -- a model drifting
    into Latin shows up as a gap between the two, not as a high number.
    """
    letters = [character for character in (text or "") if character.isalpha()]
    if not letters:
        return 0.0
    foreign = sum(1 for character in letters if not _is_arabic_script(character))
    return foreign / len(letters)


def _is_arabic_script(character):
    return any(low <= character <= high for low, high in _ARABIC_RANGES)


def _ratio(numerator, denominator):
    return numerator / denominator if denominator else 0.0
''')

In [ ]:
write("evaluation/semantic_metrics.py", r'''
"""Embedding-based metrics -- optional, and off unless asked for.

These are the only metrics here that load a model. Everything else in this
service is pure text processing that runs in milliseconds on a CPU; BERTScore
and sentence similarity pull in torch and download weights, so they are opt-in
per request rather than part of the default score.

Install them with:
    pip install -r requirements-semantic.txt

`available()` reports whether that has been done, so a caller can find out
before asking rather than by failing.
"""
import config

_models = {}


def available():
    """(usable, reason) -- reason is empty when usable."""
    try:
        import bert_score  # noqa: F401
        import sentence_transformers  # noqa: F401
    except ImportError as exc:
        return False, (f"{exc}. Install with: pip install -r requirements-semantic.txt")
    return True, ""


def compute(reference_text, hypothesis_text):
    """Meaning-level agreement between the two texts.

    bertscore_f1        token-level match through contextual embeddings, so a
                        valid synonym is credited where WER penalises it.
    semantic_similarity whole-text cosine similarity: did the message survive.
    """
    return compute_batch([(reference_text, hypothesis_text)])[0]


def compute_batch(pairs):
    """The same two scores for many (reference, hypothesis) pairs at once.

    Both models are loaded once and every pair goes through in a single
    forward pass. Scoring a benchmark report-by-report instead means reloading
    weights and running a batch of one, several hundred times over -- which is
    the difference between a minute and an afternoon.
    """
    usable, reason = available()
    if not usable:
        raise RuntimeError(reason)
    if not pairs:
        return []

    references = [reference for reference, _ in pairs]
    hypotheses = [hypothesis for _, hypothesis in pairs]
    f1_scores = _bertscore(references, hypotheses)
    similarities = _similarity(references, hypotheses)

    return [{"bertscore_f1": round(f1, 4), "semantic_similarity": round(similarity, 4)}
            for f1, similarity in zip(f1_scores, similarities)]


def _scorer():
    """One BERTScorer for the process.

    `bert_score.score()` builds a fresh scorer -- and reloads the weights --
    on every call, so calling it per report is the single slowest thing this
    service can do. The class form keeps the model.
    """
    if "bertscore" not in _models:
        from bert_score import BERTScorer

        _models["bertscore"] = BERTScorer(
            model_type=config.BERTSCORE_MODEL,
            num_layers=config.BERTSCORE_LAYERS,
            idf=False)
    return _models["bertscore"]


def _bertscore(references, hypotheses):
    _, _, f1 = _scorer().score(hypotheses, references, verbose=False)
    return [float(value) for value in f1]


def _similarity(references, hypotheses):
    from sentence_transformers import SentenceTransformer, util

    # Loading is the expensive part, so the model is kept for the process.
    if "similarity" not in _models:
        _models["similarity"] = SentenceTransformer(config.SIMILARITY_MODEL)
    model = _models["similarity"]

    reference_embeddings = model.encode(references, convert_to_tensor=True)
    hypothesis_embeddings = model.encode(hypotheses, convert_to_tensor=True)
    return [float(util.cos_sim(reference, hypothesis)[0][0])
            for reference, hypothesis in zip(reference_embeddings, hypothesis_embeddings)]
''')

In [ ]:
write("evaluation/medical_metrics.py", r'''
"""The metrics themselves.

`evaluate()` is the whole public surface: give it a hypothesis and a reference
and it returns the report described in evaluation_schema.json.

The order matters. Nothing is counted until both texts have been normalised and
their entities aligned, because comparing raw text turns a single substitution
into one addition plus one omission -- inflating two metrics while the metric
that should have caught it reads zero.
"""
from dataclasses import dataclass

import config
import general_metrics
import semantic_metrics
from extractors import ClinicalTerms, extract_measurements
from text_normalizer import tokenize

# 1.1.0 added the denominators to clinical_counts; 1.2.0 added the character
# counts and script contamination. Additive -- no metric value moved -- but
# results carry the version so the difference is never guesswork.
METRICS_VERSION = "1.2.0"

# Negation applies to findings, not to body parts: a report says "no stone",
# never "no kidney". Scoring anatomy for negation just measures how far the
# cue window happened to reach.
NEGATABLE_CATEGORIES = {"finding", "device"}


# --- Measurement alignment -------------------------------------------------
def align_measurements(reference, hypothesis):
    """Pair up measurements before judging them.

    Exact physical matches are paired first so an out-of-order but correct
    measurement is never mistaken for an error; whatever is left is paired in
    order of appearance within the same dimension. Leftovers on the reference
    side are omissions, leftovers on the hypothesis side are additions.
    """
    remaining_hypothesis = list(hypothesis)
    pairs = []
    unmatched_reference = []

    for measurement in reference:
        match = next((candidate for candidate in remaining_hypothesis
                      if measurement.same_quantity(candidate)), None)
        if match is not None:
            remaining_hypothesis.remove(match)
            pairs.append((measurement, match))
        else:
            unmatched_reference.append(measurement)

    still_unmatched = []
    for measurement in unmatched_reference:
        match = next((candidate for candidate in remaining_hypothesis
                      if candidate.dimension == measurement.dimension), None)
        if match is not None:
            remaining_hypothesis.remove(match)
            pairs.append((measurement, match))
        else:
            still_unmatched.append(measurement)

    return pairs, still_unmatched, remaining_hypothesis


def classify_measurement_error(reference, hypothesis):
    """Which counter a mismatched pair belongs to.

    Comparing physical quantities rather than unit strings means 10 mm and
    1 cm agree. When they genuinely differ, the literal tells us what went
    wrong: the same number with a different unit is a misheard unit, a
    different number with the same unit is a misheard value.
    """
    if reference.same_quantity(hypothesis):
        return set()
    errors = set()
    if reference.unit != hypothesis.unit:
        errors.add("unit")
    if reference.value != hypothesis.value:
        errors.add("number")
    if not errors:
        errors.add("number")
    return errors


# --- Comparing the two texts -----------------------------------------------
@dataclass(frozen=True)
class ConceptComparison:
    """How the two texts agreed about clinical concepts."""
    reference: dict
    hypothesis: dict
    matched: set
    only_reference: set
    only_hypothesis: set
    negation_errors: int
    negation_scored: int
    laterality_errors: int
    laterality_scored: int
    critical_errors: list


@dataclass(frozen=True)
class MeasurementComparison:
    """How the two texts agreed about measured values."""
    reference_count: int
    hypothesis_count: int
    number_errors: int
    unit_errors: int
    missing: list
    extra: list
    critical_errors: list


def _index_by_concept(mentions):
    """One entry per concept; the first mention carries its negation and side."""
    indexed = {}
    for mention in mentions:
        indexed.setdefault(mention.concept_id, mention)
    return indexed


def _compare_concepts(reference, hypothesis):
    """Negation and laterality, judged only on concepts present in both texts.

    A concept in just one text is a term-level miss, already counted by
    precision and recall; comparing its negation against nothing would mean
    nothing.
    """
    matched = set(reference) & set(hypothesis)
    negation_errors = negation_scored = 0
    laterality_errors = laterality_scored = 0
    critical_errors = []

    for concept_id in sorted(matched):
        expected, produced = reference[concept_id], hypothesis[concept_id]

        if expected.category in NEGATABLE_CATEGORIES:
            negation_scored += 1
            if expected.negated != produced.negated:
                negation_errors += 1
                critical_errors.append({
                    "type": "negation_flip", "concept": concept_id,
                    "reference": "negative" if expected.negated else "positive",
                    "prediction": "negative" if produced.negated else "positive",
                })

        if expected.laterality is not None:
            laterality_scored += 1
            if expected.laterality != produced.laterality:
                laterality_errors += 1
                critical_errors.append({
                    "type": "laterality_flip", "concept": concept_id,
                    "reference": expected.laterality,
                    "prediction": produced.laterality,
                })

    return ConceptComparison(
        reference=reference, hypothesis=hypothesis, matched=matched,
        only_reference=set(reference) - set(hypothesis),
        only_hypothesis=set(hypothesis) - set(reference),
        negation_errors=negation_errors, negation_scored=negation_scored,
        laterality_errors=laterality_errors, laterality_scored=laterality_scored,
        critical_errors=critical_errors,
    )


def _compare_measurements(reference_text, hypothesis_text):
    """Align the measurements, then judge each pair."""
    reference = extract_measurements(reference_text)
    hypothesis = extract_measurements(hypothesis_text)
    pairs, missing, extra = align_measurements(reference, hypothesis)

    number_errors = unit_errors = 0
    critical_errors = []
    for expected, produced in pairs:
        kinds = classify_measurement_error(expected, produced)
        number_errors += "number" in kinds
        unit_errors += "unit" in kinds
        if kinds:
            critical_errors.append({
                "type": "measurement_mismatch", "concept": "measurement",
                "reference": _format(expected), "prediction": _format(produced),
            })
    for expected in missing:
        critical_errors.append({
            "type": "measurement_omission", "concept": "measurement",
            "reference": _format(expected), "prediction": None,
        })

    return MeasurementComparison(
        reference_count=len(reference), hypothesis_count=len(hypothesis),
        number_errors=number_errors, unit_errors=unit_errors,
        missing=missing, extra=extra, critical_errors=critical_errors,
    )


def _dropped_critical_concepts(concepts):
    """Concepts the reference stated with a negation or a side that the output
    dropped entirely -- content a reader would have acted on."""
    return [
        concept_id for concept_id in sorted(concepts.only_reference)
        if concepts.reference[concept_id].negated
        or concepts.reference[concept_id].laterality is not None
    ]


def _term_scores(true_positives, false_positives, false_negatives):
    """Precision, recall and F1 over concept identities."""
    precision = _ratio(true_positives, true_positives + false_positives)
    recall = _ratio(true_positives, true_positives + false_negatives)
    return precision, recall, _ratio(2 * precision * recall, precision + recall)


# --- The report ------------------------------------------------------------
def evaluate(hypothesis_text, reference_text, terms=None, include_semantic=False):
    """Score one report against its reference.

    `hypothesis_text` is what the pipeline produced; `reference_text` is the
    radiologist-verified version. `include_semantic` adds the embedding
    metrics, which load a model and are therefore opt-in.
    """
    terms = terms or ClinicalTerms()

    concepts = _compare_concepts(
        _index_by_concept(terms.find(tokenize(reference_text))),
        _index_by_concept(terms.find(tokenize(hypothesis_text))))
    measurements = _compare_measurements(reference_text, hypothesis_text)

    dropped = _dropped_critical_concepts(concepts)
    critical_omissions = len(measurements.missing) + len(dropped)
    unsupported_additions = len(measurements.extra) + len(concepts.only_hypothesis)

    # How much there was to get wrong. Reported alongside the errors because a
    # rate cannot be re-derived across a batch without it: summing rates is not
    # the same as a rate over summed counts, and only the latter is meaningful
    # when reports vary in length.
    omission_scored = measurements.reference_count + len(dropped)
    addition_scored = measurements.hypothesis_count + len(concepts.hypothesis)

    true_positives = len(concepts.matched)
    false_positives = len(concepts.only_hypothesis)
    false_negatives = len(concepts.only_reference)
    precision, recall, f1 = _term_scores(true_positives, false_positives, false_negatives)

    wer = general_metrics.word_error_rate(reference_text, hypothesis_text)
    cer = general_metrics.character_error_rate(reference_text, hypothesis_text)

    # A degenerate output needs a human look even when every clinical counter
    # reads zero -- a model repeating one plausible sentence forever produces
    # no negation, laterality or number error at all.
    repetition = general_metrics.repetition_score(hypothesis_text)
    length_ratio = general_metrics.hallucination_ratio(reference_text, hypothesis_text)
    degenerate = [
        name for name, tripped in (
            ("repetition", repetition >= config.MAX_REPETITION),
            ("length_ratio", length_ratio >= config.MAX_LENGTH_RATIO
                             or (reference_text.strip() and length_ratio <= config.MIN_LENGTH_RATIO)),
        ) if tripped
    ]

    reasons = degenerate + [name for name, count in (
        ("negation_errors", concepts.negation_errors),
        ("laterality_errors", concepts.laterality_errors),
        ("number_errors", measurements.number_errors),
        ("unit_errors", measurements.unit_errors),
        ("critical_omissions", critical_omissions),
        ("unsupported_additions", unsupported_additions),
    ) if count]

    report = {
        "general": {
            "wer": round(wer.rate, 4),
            "cer": round(cer.rate, 4),
            "substitutions": wer.substitutions,
            "insertions": wer.insertions,
            "deletions": wer.deletions,
            "reference_words": wer.reference_length,
            "hypothesis_words": len(tokenize(hypothesis_text)),
            # Character-level counts, so a batch can report a corpus CER --
            # total edits over total characters -- rather than a mean of
            # per-report rates, which weights a one-line report like a page.
            "character_errors": cer.errors,
            "reference_chars": cer.reference_length,
            "chrf": round(general_metrics.chrf(reference_text, hypothesis_text), 4),
            "hallucination_ratio": round(length_ratio, 4),
            "repetition_score": round(repetition, 4),
            "punctuation_f1": round(
                general_metrics.punctuation_f1(reference_text, hypothesis_text), 4),
            # Reported as a pair on purpose. This corpus code-switches English
            # radiology terms deliberately, so the hypothesis figure alone says
            # nothing; the gap between the two is the part that does.
            "script_contamination": round(
                general_metrics.script_contamination(hypothesis_text), 4),
            "reference_script_contamination": round(
                general_metrics.script_contamination(reference_text), 4),
        },
        "clinical_counts": {
            "reference_entities": len(concepts.reference),
            "matched_entities": true_positives,
            "true_positive_terms": true_positives,
            "false_positive_terms": false_positives,
            "false_negative_terms": false_negatives,
            "reference_measurements": measurements.reference_count,
            "hypothesis_measurements": measurements.hypothesis_count,
            "negation_errors": concepts.negation_errors,
            "laterality_errors": concepts.laterality_errors,
            "number_errors": measurements.number_errors,
            "unit_errors": measurements.unit_errors,
            "critical_omissions": critical_omissions,
            "unsupported_additions": unsupported_additions,
            # Denominators, so every rate above can be rebuilt over a batch.
            "negation_scored": concepts.negation_scored,
            "laterality_scored": concepts.laterality_scored,
            "critical_omission_scored": omission_scored,
            "unsupported_addition_scored": addition_scored,
        },
        "clinical_metrics": {
            "medical_term_precision": round(precision, 4),
            "medical_term_recall": round(recall, 4),
            "medical_term_f1": round(f1, 4),
            "negation_error_rate": round(
                _ratio(concepts.negation_errors, concepts.negation_scored), 4),
            "laterality_error_rate": round(
                _ratio(concepts.laterality_errors, concepts.laterality_scored), 4),
            "number_error_rate": round(
                _ratio(measurements.number_errors, measurements.reference_count), 4),
            "unit_error_rate": round(
                _ratio(measurements.unit_errors, measurements.reference_count), 4),
            "critical_omission_rate": round(_ratio(critical_omissions, omission_scored), 4),
            "unsupported_addition_rate": round(_ratio(unsupported_additions, addition_scored), 4),
        },
        "critical_errors": concepts.critical_errors + measurements.critical_errors,
        "requires_medical_review": bool(reasons),
        "review_reasons": reasons,
        "evaluation": {
            "metrics_version": METRICS_VERSION,
            "terms_version": terms.version,
            "terms_sha": terms.sha,
        },
    }

    if include_semantic:
        report["semantic"] = semantic_metrics.compute(reference_text, hypothesis_text)
    return report


def _ratio(numerator, denominator):
    """Rates are 0.0 when nothing was scorable, never a division error."""
    return numerator / denominator if denominator else 0.0


def _format(measurement):
    number = int(measurement.value) if measurement.value.is_integer() else measurement.value
    return f"{number} {measurement.unit}" if measurement.unit else str(number)
''')

In [ ]:
write("evaluation/evaluate_results.py", r'''
"""Batch scoring from the command line -- no server involved.

Reads a manifest of report pairs, scores each one, and writes a result file per
report plus a summary across the batch.

    python evaluate_results.py manifest.json --out results/

The manifest is a JSON array; `hypothesis`/`reference` may be inline text or a
path to a .txt file:

    [
      {"asset_id": "DPM89130", "model": "whisper-large-v3",
       "hypothesis": "out/DPM89130.txt", "reference": "truth/DPM89130.txt"}
    ]

Aggregate rates are computed from summed counts, never by averaging per-report
rates: a short report with one negation would otherwise turn a single error
into a 100% rate and drown out everything else.
"""
import argparse
import json
import pathlib
import sys

import config
from extractors import ClinicalTerms
from medical_metrics import METRICS_VERSION, evaluate

COUNT_FIELDS = [
    "reference_entities", "matched_entities", "true_positive_terms",
    "false_positive_terms", "false_negative_terms",
    "reference_measurements", "hypothesis_measurements",
    "negation_errors", "laterality_errors", "number_errors", "unit_errors",
    "critical_omissions", "unsupported_additions",
    "negation_scored", "laterality_scored",
    "critical_omission_scored", "unsupported_addition_scored",
]

# Per-report text metrics that are scores rather than counts, so they aggregate
# as a distribution rather than a sum. Which end of the distribution matters is
# not the same for all of them: chrF and punctuation F1 describe typical
# quality, while repetition and length ratio are failure detectors -- their mean
# is near zero even when a model loops on one report in twenty, so the tail is
# the number worth reading.
TEXT_FIELDS = {
    "cer": "mean",
    "chrf": "mean",
    "punctuation_f1": "mean",
    "script_contamination": "mean",
    "reference_script_contamination": "mean",
    "repetition_score": "tail",
    "hallucination_ratio": "tail",
}

# Edit counts summed across the batch, which is what a corpus WER is made of.
EDIT_FIELDS = [
    "substitutions", "insertions", "deletions",
    "reference_words", "hypothesis_words",
    "character_errors", "reference_chars",
]


def _corpus_rates(results):
    """Corpus WER and CER: total edits over total reference length.

    This is the headline number, and it is not the mean of the per-report WERs.
    A mean weights a one-line report exactly like a full page, so a model that
    fails on the short ones looks worse than it is -- and one that fails on the
    long ones looks better. `wer_p50` and friends describe the spread around
    this; they do not replace it.
    """
    totals = {field: 0 for field in EDIT_FIELDS}
    for result in results:
        for field in EDIT_FIELDS:
            totals[field] += result["general"].get(field, 0)

    word_errors = totals["substitutions"] + totals["insertions"] + totals["deletions"]
    reference_words = totals["reference_words"]
    return {
        **totals,
        "corpus_wer": round(_ratio(word_errors, reference_words), 4),
        "corpus_cer": round(_ratio(totals["character_errors"], totals["reference_chars"]), 4),
        # Which of the three the errors actually are. Insertions matter most in
        # a medical transcript: they are the fabrication signal, not just noise.
        "substitution_rate": round(_ratio(totals["substitutions"], reference_words), 4),
        "insertion_rate": round(_ratio(totals["insertions"], reference_words), 4),
        "deletion_rate": round(_ratio(totals["deletions"], reference_words), 4),
    }


def _read(value, base):
    """Inline text, or the contents of a file path relative to the manifest."""
    candidate = base / value
    if len(value) < 260 and candidate.is_file():
        return candidate.read_text(encoding="utf-8")
    return value


def score_manifest(manifest_path, terms):
    manifest_path = pathlib.Path(manifest_path)
    entries = json.loads(manifest_path.read_text(encoding="utf-8"))
    base = manifest_path.parent

    results = []
    for entry in entries:
        result = evaluate(_read(entry["hypothesis"], base),
                          _read(entry["reference"], base), terms)
        results.append({
            "asset_id": entry["asset_id"],
            "model": entry["model"],
            "pipeline": entry.get("pipeline"),
            "model_version": entry.get("model_version"),
            **result,
        })
    return results


def _reliability(results):
    """Distribution metrics -- the ones that only exist across many reports.

    A mean WER hides the shape of the failures: a model can look acceptable on
    average while collapsing completely on one report in twenty. These say how
    often, and how badly.
    """
    wers = sorted(result["general"]["wer"] for result in results)
    total = len(wers)
    if not total:
        return {}

    def percentile(fraction):
        return round(wers[min(int(fraction * total), total - 1)], 4)

    perfect = sum(1 for wer in wers if wer == 0)
    return {
        # Share of reports with any error at all.
        "ser": round(_ratio(total - perfect, total), 4),
        "wer_p50": percentile(0.50),
        "wer_p90": percentile(0.90),
        "wer_p95": percentile(0.95),
        # In a post-edit workflow this is also the "accepted unchanged" rate:
        # a zero WER means the radiologist altered nothing.
        "pct_perfect": round(_ratio(perfect, total), 4),
        "pct_catastrophic": round(
            _ratio(sum(1 for wer in wers if wer >= config.CATASTROPHIC_WER), total), 4),
        "empty_output_rate": round(_ratio(
            sum(1 for r in results if r["general"]["hypothesis_words"] == 0), total), 4),
    }


def _text_distribution(results):
    """Corpus-level view of the per-report text scores.

    WER gets percentiles because it is the headline; these get the same
    treatment for the same reason -- a model that is fine on average and
    collapses on one report in twenty is not fine, and only the tail says so.
    """
    summary = {}
    for field, emphasis in TEXT_FIELDS.items():
        values = sorted(result["general"][field] for result in results
                        if field in result["general"])
        if not values:
            continue
        summary[f"{field}_mean"] = round(sum(values) / len(values), 4)
        if emphasis == "tail":
            summary[f"{field}_p95"] = round(values[min(int(0.95 * len(values)), len(values) - 1)], 4)
            summary[f"{field}_max"] = round(values[-1], 4)
    return summary


def summarize(results, terms):
    """Per-model totals, with rates recomputed from the summed counts."""
    by_model = {}
    for result in results:
        bucket = by_model.setdefault(result["model"], {
            "model": result["model"], "reports": 0,
            "requires_medical_review": 0,
            **{field: 0 for field in COUNT_FIELDS},
        })
        bucket["reports"] += 1
        bucket["requires_medical_review"] += int(result["requires_medical_review"])
        for field in COUNT_FIELDS:
            bucket[field] += result["clinical_counts"][field]

    for model, bucket in by_model.items():
        model_results = [r for r in results if r["model"] == model]
        bucket["corpus"] = _corpus_rates(model_results)
        bucket["reliability"] = _reliability(model_results)
        bucket["text"] = _text_distribution(model_results)
        measurements = bucket["reference_measurements"]
        produced = bucket["true_positive_terms"] + bucket["false_positive_terms"]
        expected = bucket["true_positive_terms"] + bucket["false_negative_terms"]
        precision = _ratio(bucket["true_positive_terms"], produced)
        recall = _ratio(bucket["true_positive_terms"], expected)
        # Every rate is errors over what there was to get wrong, summed across
        # the batch -- never the mean of the per-report rates. One negation
        # error in a two-sentence report is a 100% rate, and averaging that in
        # would drown out a hundred correct ones.
        bucket["rates"] = {
            "medical_term_precision": round(precision, 4),
            "medical_term_recall": round(recall, 4),
            "medical_term_f1": round(_ratio(2 * precision * recall, precision + recall), 4),
            "negation_error_rate": round(
                _ratio(bucket["negation_errors"], bucket["negation_scored"]), 4),
            "laterality_error_rate": round(
                _ratio(bucket["laterality_errors"], bucket["laterality_scored"]), 4),
            "number_error_rate": round(_ratio(bucket["number_errors"], measurements), 4),
            "unit_error_rate": round(_ratio(bucket["unit_errors"], measurements), 4),
            "critical_omission_rate": round(
                _ratio(bucket["critical_omissions"], bucket["critical_omission_scored"]), 4),
            "unsupported_addition_rate": round(
                _ratio(bucket["unsupported_additions"], bucket["unsupported_addition_scored"]), 4),
            "review_rate": round(_ratio(bucket["requires_medical_review"], bucket["reports"]), 4),
        }
    return {
        "models": sorted(by_model.values(), key=lambda b: b["model"]),
        "reports": len(results),
        "evaluation": {
            "metrics_version": METRICS_VERSION,
            "terms_version": terms.version,
            "terms_sha": terms.sha,
        },
    }


def _ratio(numerator, denominator):
    return numerator / denominator if denominator else 0.0


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("manifest", help="JSON array of report pairs")
    parser.add_argument("--out", default="results", help="output directory")
    parser.add_argument("--terms", default=None, help="path to clinical_terms.json")
    args = parser.parse_args(argv)

    terms = ClinicalTerms(args.terms)
    results = score_manifest(args.manifest, terms)

    out_dir = pathlib.Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)
    for result in results:
        # One file per report per model, so nothing overwrites anything else.
        name = f"{result['asset_id']}__{result['model'].replace('/', '-')}.json"
        (out_dir / name).write_text(
            json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")

    summary = summarize(results, terms)
    (out_dir / "summary.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"scored {len(results)} report(s) -> {out_dir}")
    for bucket in summary["models"]:
        rates = bucket["rates"]
        reliability = bucket["reliability"]
        print(f"  {bucket['model']:28} reports={bucket['reports']:4} "
              f"WER={bucket['corpus']['corpus_wer']:.3f} "
              f"F1={rates['medical_term_f1']:.3f} "
              f"num_err={rates['number_error_rate']:.3f} "
              f"review={rates['review_rate']:.0%} "
              f"wer_p90={reliability['wer_p90']:.3f} "
              f"catastrophic={reliability['pct_catastrophic']:.0%}")
    return 0


if __name__ == "__main__":
    sys.exit(main())
''')

In [ ]:
write("evaluation/clinical_terms.json", r'''
{
  "version": "2026-08-31",
  "scope": "abdominal / pelvic ultrasound",
  "note": "Seed list built from observed dictations. Expand from RadLex (English backbone, freely licensed) plus corpus frequency mining; variants are matched against normalizer output, so write them post-normalization (no ZWNJ, ASCII digits, lowercase).",
  "concepts": [
    {"id": "kidney", "category": "anatomy", "variants": ["kidney", "kidneys", "renal", "کلیه", "کلیه ها", "کیدنی"]},
    {"id": "ureter", "category": "anatomy", "variants": ["ureter", "ureteral", "حالب", "یورتر"]},
    {"id": "uvj", "category": "anatomy", "variants": ["uvj", "vesicoureteric junction", "ureterovesical junction"]},
    {"id": "urinary_bladder", "category": "anatomy", "variants": ["urinary bladder", "bladder", "مثانه"]},
    {"id": "gallbladder", "category": "anatomy", "variants": ["gallbladder", "gall bladder", "کیسه صفرا"]},
    {"id": "common_bile_duct", "category": "anatomy", "variants": ["common bile duct", "cbd", "مجرای صفراوی"]},
    {"id": "liver", "category": "anatomy", "variants": ["liver", "hepatic", "کبد"]},
    {"id": "spleen", "category": "anatomy", "variants": ["spleen", "splenic", "طحال", "اسپلین"]},
    {"id": "prostate", "category": "anatomy", "variants": ["prostate", "prostatic", "پروستات"]},
    {"id": "pancreas", "category": "anatomy", "variants": ["pancreas", "pancreatic", "لوزالمعده", "پانکراس"]},
    {"id": "retroperitoneum", "category": "anatomy", "variants": ["retroperitoneum", "retroperitoneal", "رتروپریتوئن"]},
    {"id": "umbilicus", "category": "anatomy", "variants": ["umbilicus", "umbilical", "ناف"]},
    {"id": "abdomen", "category": "anatomy", "variants": ["abdomen", "abdominal", "شکم"]},
    {"id": "pelvis", "category": "anatomy", "variants": ["pelvis", "pelvic", "لگن"]},

    {"id": "calculus", "category": "finding", "variants": ["stone", "stones", "calculus", "calculi", "سنگ"]},
    {"id": "hydronephrosis", "category": "finding", "variants": ["hydronephrosis", "hydroureteronephrosis", "هیدرونفروز"]},
    {"id": "dilatation", "category": "finding", "variants": ["dilatation", "dilation", "dilated", "اتساع", "گشاد"]},
    {"id": "cyst", "category": "finding", "variants": ["cyst", "cysts", "cystic", "کیست"]},
    {"id": "mass", "category": "finding", "variants": ["mass", "masses", "tumor", "tumour", "توده"]},
    {"id": "adenopathy", "category": "finding", "variants": ["adenopathy", "lymphadenopathy", "آدنوپاتی"]},
    {"id": "free_fluid", "category": "finding", "variants": ["free fluid", "ascites", "مایع آزاد", "آسیت"]},
    {"id": "fluid_collection", "category": "finding", "variants": ["fluid collection", "collection", "تجمع مایع"]},
    {"id": "cholecystitis", "category": "finding", "variants": ["cholecystitis", "کوله سیستیت"]},
    {"id": "echogenic_debris", "category": "finding", "variants": ["echogenic debris", "debris", "دبری"]},
    {"id": "hematuria", "category": "finding", "variants": ["hematuria", "haematuria", "هماچوری"]},
    {"id": "infection", "category": "finding", "variants": ["infection", "infective", "عفونت"]},
    {"id": "fatty_liver", "category": "finding", "variants": ["fatty liver", "hepatic steatosis", "steatosis", "کبد چرب"]},
    {"id": "bph", "category": "finding", "variants": ["bph", "benign prostatic hyperplasia", "benign prostatic hypertrophy"]},
    {"id": "outlet_obstruction", "category": "finding", "variants": ["outlet obstruction", "bladder outlet obstruction", "انسداد"]},
    {"id": "wall_thickening", "category": "finding", "variants": ["wall thickening", "thick wall", "thickened wall", "ضخیم شدگی"]},
    {"id": "septa", "category": "finding", "variants": ["septa", "septation", "septations", "سپتا"]},
    {"id": "parenchymal_echogenicity", "category": "finding", "variants": ["parenchymal echogenicity", "echogenicity", "اکوژنیسیته"]},
    {"id": "hepatomegaly", "category": "finding", "variants": ["hepatomegaly", "بزرگی کبد"]},
    {"id": "splenomegaly", "category": "finding", "variants": ["splenomegaly", "بزرگی طحال"]},

    {"id": "vp_shunt", "category": "device", "variants": ["vp shunt", "ventriculoperitoneal shunt", "شنت"]},
    {"id": "psa", "category": "workup", "variants": ["psa"]},
    {"id": "fna", "category": "workup", "variants": ["fna", "fine needle aspiration"]},
    {"id": "urinalysis", "category": "workup", "variants": ["urinalysis", "u a", "ua"]},
    {"id": "pvr", "category": "workup", "variants": ["pvr", "post void residual", "postvoid residual"]},
    {"id": "ct_scan", "category": "workup", "variants": ["ct scan", "ct", "سی تی"]},
    {"id": "follow_up", "category": "workup", "variants": ["follow up", "followup", "پیگیری"]}
  ]
}
''')

### The models

From `stt/app/` — one class per architecture, and a registry naming every
model. Adding a model is one registry entry; nothing below names a model.

In [ ]:
write("stt/app/config.py", r'''
"""Central configuration, loaded from the environment or a .env file."""
import os

import torch
from dotenv import load_dotenv

load_dotenv()  # reads .env if present; no-op otherwise

# --- Runtime ---
HOST = os.getenv("HOST", "0.0.0.0")
PORT = int(os.getenv("PORT", "8000"))
ALLOWED_ORIGINS = os.getenv("ALLOWED_ORIGINS", "*").split(",")
DEVICE = os.getenv("DEVICE") or ("cuda" if torch.cuda.is_available() else "cpu")

# --- Audio ---
TARGET_SAMPLE_RATE = 16000

# --- Languages ---
# Codes accepted by POST /transcribe's `language` field, mapped to a display
# name. Each model class maps these onto whatever codes it actually needs
# (see WHISPER_LANGUAGE_NAMES / SEAMLESS_LANGUAGE_CODES in model.py).
SUPPORTED_LANGUAGES = {
    "fa": "Persian",
    "en": "English",
}
DEFAULT_LANGUAGE = os.getenv("DEFAULT_LANGUAGE", "fa")

# --- Models ---
# A registry key to load at startup, instead of waiting for the first request.
PRELOAD_MODEL = os.getenv("PRELOAD_MODEL") or None

# Ordered best -> worst by word error rate on clinic-realistic noise. The UI
# dropdowns show this order, so it is a ranking, not an arbitrary list --
# don't reorder without a benchmark run to justify it.
MODEL_REGISTRY = {
    "seamless": {
        "type": "seamless_v2",
        "model_id": "facebook/seamless-m4t-v2-large",
        "tgt_lang": "pes",
    },
    "whisper": {
        "type": "whisper",
        "model_id": "nezamisafa/whisper-persian-v4",
    },
    "seamless-medium": {
        "type": "seamless_v1",
        "model_id": "facebook/hf-seamless-m4t-medium",
        "tgt_lang": "pes",
    },
    "whisper-halakoo": {
        "type": "whisper",
        "model_id": "MohammadReza-Halakoo/persian-whisper-large-v3-10-percent-17-0-one-epoch",
    },
    "mms-all": {
        "type": "mms",
        "model_id": "facebook/mms-1b-all",
        "target_lang": "fas",
    },
    "mms-fl102": {
        "type": "mms",
        "model_id": "facebook/mms-1b-fl102",
        "target_lang": "fas",
    },
    "whisper-vhdm": {
        "type": "whisper",
        "model_id": "vhdm/whisper-large-fa-v1",
    },
    "wav2vec2-xlsr53": {
        "type": "ctc",
        "model_id": "jonatasgrosman/wav2vec2-large-xlsr-53-persian",
    },
    "whisper-large-v3": {
        "type": "whisper",
        "model_id": "openai/whisper-large-v3",
    },
    "whisper-large-v3-turbo": {
        "type": "whisper",
        "model_id": "openai/whisper-large-v3-turbo",
    },
}
''')

In [ ]:
write("stt/app/model.py", r'''
"""Model loading, swapping and transcription.

Each supported architecture is a subclass of ``BaseSTTModel``. A new model is
added by writing one subclass, registering it in ``_MODEL_TYPES`` and adding an
entry to ``config.MODEL_REGISTRY`` — nothing in the API layer changes.
"""

import gc
import threading
from abc import ABC, abstractmethod

import numpy as np
import torch
from transformers import (
    AutoProcessor,
    SeamlessM4TModel,
    SeamlessM4Tv2Model,
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)

from app import config

# Map the API's simple language codes onto what each model family expects.
WHISPER_LANGUAGE_NAMES = {"fa": "persian", "en": "english"}
SEAMLESS_LANGUAGE_CODES = {"fa": "pes", "en": "eng"}


def resample(audio, sr, target_sr=config.TARGET_SAMPLE_RATE):
    """Resample a mono float32 array to ``target_sr`` if needed."""
    if sr == target_sr:
        return audio
    import torchaudio

    tensor = torch.from_numpy(np.asarray(audio, np.float32)).unsqueeze(0)
    out = torchaudio.functional.resample(tensor, sr, target_sr)
    return out.squeeze(0).numpy()


class BaseSTTModel(ABC):
    """An STT model that can be loaded into memory and run on audio."""

    def __init__(self, model_id, device):
        self.model_id = model_id
        self.device = device
        self._model = None
        self._processor = None

    @abstractmethod
    def load(self):
        """Pull weights and processor into memory on ``self.device``."""

    @abstractmethod
    def transcribe(self, audio, sr, language=None):
        """Return the transcription of a mono float32 array sampled at ``sr``.

        ``language`` is one of ``config.SUPPORTED_LANGUAGES`` (e.g. "fa", "en"),
        or ``None`` to use the model's own default behaviour.
        """

    def unload(self):
        """Release references and free GPU memory."""
        self._model = None
        self._processor = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


class WhisperModel(BaseSTTModel):
    """Whisper-family conditional generation model."""

    def load(self):
        dtype = torch.float16 if self.device == "cuda" else torch.float32
        self._processor = WhisperProcessor.from_pretrained(self.model_id)
        self._model = (
            WhisperForConditionalGeneration.from_pretrained(self.model_id, torch_dtype=dtype)
            .to(self.device)
            .eval()
        )
        self._model.generation_config.forced_decoder_ids = None

    def transcribe(self, audio, sr, language=None):
        audio = resample(audio, sr)
        features = self._processor(
            audio, sampling_rate=config.TARGET_SAMPLE_RATE, return_tensors="pt"
        ).input_features.to(self.device)
        if self.device == "cuda":
            features = features.half()

        forced_decoder_ids = None
        if language:
            name = WHISPER_LANGUAGE_NAMES.get(language, language)
            forced_decoder_ids = self._processor.get_decoder_prompt_ids(
                language=name, task="transcribe"
            )

        with torch.no_grad():
            ids = self._model.generate(features, forced_decoder_ids=forced_decoder_ids)
        return self._processor.batch_decode(ids, skip_special_tokens=True)[0]


class SeamlessV2Model(BaseSTTModel):
    """SeamlessM4T v2 speech-to-text model."""

    def __init__(self, model_id, device, tgt_lang="pes"):
        super().__init__(model_id, device)
        self.tgt_lang = tgt_lang

    def load(self):
        self._processor = AutoProcessor.from_pretrained(self.model_id)
        self._model = SeamlessM4Tv2Model.from_pretrained(self.model_id).to(self.device).eval()

    def transcribe(self, audio, sr, language=None):
        audio = resample(audio, sr)
        inputs = self._processor(
            audio=audio, sampling_rate=config.TARGET_SAMPLE_RATE, return_tensors="pt"
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        tgt_lang = SEAMLESS_LANGUAGE_CODES.get(language, language) if language else self.tgt_lang
        with torch.no_grad():
            out = self._model.generate(
                **inputs, tgt_lang=tgt_lang, generate_speech=False
            )
        seqs = out.sequences if hasattr(out, "sequences") else out
        return self._processor.tokenizer.batch_decode(seqs, skip_special_tokens=True)[0]


class SeamlessV1Model(BaseSTTModel):
    """SeamlessM4T v1 speech-to-text model (e.g. hf-seamless-m4t-medium).

    v1's ``generate(..., generate_speech=False)`` returns token ids directly
    (no ``.sequences`` wrapper like v2), so decoding differs slightly from
    ``SeamlessV2Model``.
    """

    def __init__(self, model_id, device, tgt_lang="pes"):
        super().__init__(model_id, device)
        self.tgt_lang = tgt_lang

    def load(self):
        self._processor = AutoProcessor.from_pretrained(self.model_id)
        self._model = SeamlessM4TModel.from_pretrained(self.model_id).to(self.device).eval()

    def transcribe(self, audio, sr, language=None):
        audio = resample(audio, sr)
        inputs = self._processor(
            audios=audio, sampling_rate=config.TARGET_SAMPLE_RATE, return_tensors="pt"
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        tgt_lang = SEAMLESS_LANGUAGE_CODES.get(language, language) if language else self.tgt_lang
        with torch.no_grad():
            out = self._model.generate(**inputs, tgt_lang=tgt_lang, generate_speech=False)
        tokens = out[0] if isinstance(out, (list, tuple)) else out
        return self._processor.decode(tokens.squeeze().tolist(), skip_special_tokens=True)


class CTCModel(BaseSTTModel):
    """Plain wav2vec2-family CTC model (no language adapter)."""

    def load(self):
        self._processor = Wav2Vec2Processor.from_pretrained(self.model_id)
        self._model = Wav2Vec2ForCTC.from_pretrained(self.model_id).to(self.device).eval()

    def transcribe(self, audio, sr, language=None):
        audio = resample(audio, sr)
        inputs = self._processor(
            audio, sampling_rate=config.TARGET_SAMPLE_RATE, return_tensors="pt"
        )
        input_values = inputs.input_values.to(self.device)
        with torch.no_grad():
            logits = self._model(input_values).logits
        ids = torch.argmax(logits, dim=-1)
        return self._processor.batch_decode(ids)[0]


class MMSModel(CTCModel):
    """Meta MMS CTC model — same as ``CTCModel`` but loads a target-language
    adapter first (MMS ships one shared backbone with per-language adapter
    weights; see https://huggingface.co/facebook/mms-1b-all).
    """

    def __init__(self, model_id, device, target_lang="fas"):
        super().__init__(model_id, device)
        self.target_lang = target_lang

    def load(self):
        self._processor = Wav2Vec2Processor.from_pretrained(self.model_id, target_lang=self.target_lang)
        self._model = (
            Wav2Vec2ForCTC.from_pretrained(self.model_id, target_lang=self.target_lang)
            .to(self.device)
            .eval()
        )
        self._model.load_adapter(self.target_lang)


_MODEL_TYPES = {
    "whisper": WhisperModel,
    "seamless_v2": SeamlessV2Model,
    "seamless_v1": SeamlessV1Model,
    "ctc": CTCModel,
    "mms": MMSModel,
}


def build_model(key):
    """Instantiate (without loading) the model registered under ``key``."""
    if key not in config.MODEL_REGISTRY:
        raise KeyError(key)
    spec = dict(config.MODEL_REGISTRY[key])
    model_type = spec.pop("type")
    model_id = spec.pop("model_id")
    cls = _MODEL_TYPES[model_type]
    return cls(model_id=model_id, device=config.DEVICE, **spec)


class ModelManager:
    """Holds at most one loaded model and serializes access to it.

    A single lock guards both loading and transcription so that concurrent
    requests cannot swap the model out from under an in-flight transcription.
    """

    def __init__(self):
        self._current_key = None
        self._current_model = None
        self._lock = threading.Lock()

    def available(self):
        """Return the list of registered model keys."""
        return list(config.MODEL_REGISTRY.keys())

    @property
    def loaded(self):
        """Return the key of the loaded model, or ``None``."""
        return self._current_key

    def load(self, key):
        """Load ``key`` into memory, unloading any currently loaded model."""
        if key not in config.MODEL_REGISTRY:
            raise KeyError(key)
        with self._lock:
            if self._current_key == key:
                return
            if self._current_model is not None:
                self._current_model.unload()
                self._current_model = None
                self._current_key = None
            model = build_model(key)
            model.load()
            self._current_model = model
            self._current_key = key

    def transcribe(self, audio, sr, language=None):
        """Transcribe audio with the loaded model, erroring if none is loaded."""
        with self._lock:
            if self._current_model is None:
                raise RuntimeError("no model loaded")
            return self._current_model.transcribe(audio, sr, language=language)

    def unload(self):
        """Unload the current model and free its memory. No-op if none loaded."""
        with self._lock:
            if self._current_model is not None:
                self._current_model.unload()
                self._current_model = None
                self._current_key = None
''')

### The harness

From `benchmark/` — windowing, per-GPU scheduling, scoring and the leaderboard.

In [ ]:
write("benchmark/settings.py", r'''
"""Benchmark knobs, from the environment or `benchmark/.env`.

Deliberately **not** called `config.py`. `bridge.py` puts `evaluation/` on
`sys.path` so this module can reuse the real metrics, and `evaluation/` has a
top-level `config.py` of its own -- two files with that name on the same path
means whichever was inserted first silently wins. The name is the fix.
"""
import os
import pathlib

from dotenv import load_dotenv

load_dotenv(pathlib.Path(__file__).parent / ".env")

REPO_ROOT = pathlib.Path(__file__).resolve().parent.parent

# --- Long audio ---
# Whisper's encoder takes a fixed 30-second window: hand it four minutes of
# dictation and it transcribes the first thirty seconds and silently drops the
# rest. Every model here is therefore fed the same overlapping windows, so a
# long-audio penalty never gets mistaken for a model being worse.
WINDOW_SEC = float(os.getenv("WINDOW_SEC", "28"))
OVERLAP_SEC = float(os.getenv("OVERLAP_SEC", "3"))

# When two consecutive windows end and begin with the same words, that is the
# overlap being transcribed twice. Look back at most this many words for it.
MAX_STITCH_OVERLAP_WORDS = int(os.getenv("MAX_STITCH_OVERLAP_WORDS", "40"))

# --- Runtime ---
TARGET_SAMPLE_RATE = 16000
DEFAULT_LANGUAGE = os.getenv("BENCHMARK_LANGUAGE", "fa")

# Comma-separated torch devices, or "auto" for every visible GPU (falling back
# to CPU). On Kaggle's dual T4 this resolves to cuda:0,cuda:1 and one replica of
# the model is loaded per GPU, with the audio split between them.
DEVICES = os.getenv("BENCHMARK_DEVICES", "auto")

# Models to run when the caller names none. Keys come from stt's MODEL_REGISTRY.
DEFAULT_MODELS = [
    key for key in os.getenv("BENCHMARK_MODELS", "whisper,seamless").split(",") if key.strip()
]

# --- Output ---
def _default_out_dir():
    """Somewhere writable, which is not always next to the code.

    A Kaggle notebook usually reads the repo from `/kaggle/input`, a read-only
    mount -- so the obvious default of "beside the source" fails at mkdir, at
    the end of a run, after the GPU time has already been spent. `/kaggle/working`
    is the writable half of that runtime and the only directory Kaggle keeps.
    """
    explicit = os.getenv("BENCHMARK_OUT")
    if explicit:
        return pathlib.Path(explicit)
    kaggle_working = pathlib.Path("/kaggle/working")
    if kaggle_working.is_dir():
        return kaggle_working / "benchmark_results"
    return REPO_ROOT / "benchmark" / "results"


OUT_DIR = _default_out_dir()

# Where the sibling modules live. Set these if the benchmark is run from a copy
# of the repo that has been rearranged (a Kaggle dataset mount, say).
EVALUATION_DIR = pathlib.Path(os.getenv("EVALUATION_DIR", str(REPO_ROOT / "evaluation")))
STT_DIR = pathlib.Path(os.getenv("STT_DIR", str(REPO_ROOT / "stt")))

# Optional: score through a running evaluation service instead of in-process.
EVALUATION_URL = os.getenv("EVALUATION_URL") or None
''')

In [ ]:
write("benchmark/bridge.py", r'''
"""The one place the benchmark reaches into the other modules.

Every other service in this repo is import-isolated: it talks to its neighbours
over HTTP and never imports them. A benchmark cannot honour that -- comparing
ten models over hundreds of recordings means loading model weights in-process
(no HTTP round trip per chunk) and calling the metrics directly (no server to
stand up on a Kaggle runtime). So the coupling exists, and it is confined here:
delete this file and nothing else in `benchmark/` knows the other modules
exist.

Two rules keep it from spreading:

* `evaluation/` is imported for the metrics only -- never reimplemented. If a
  metric changes there, the benchmark changes with it.
* `stt/` is imported for its model classes only. Adding a model to
  `stt/app/config.py` is enough to make it benchmarkable; nothing here lists
  models by name.
"""
import sys

import settings


def _ensure_on_path(directory):
    """Put a sibling module's directory on `sys.path`, once.

    Appended rather than inserted: `benchmark/` must keep winning for its own
    module names, whatever the siblings happen to call theirs.
    """
    path = str(directory)
    if path not in sys.path:
        sys.path.append(path)


# --- evaluation ------------------------------------------------------------
# Imported eagerly: pure text processing, no weights, no torch. Keeping it
# eager means `import scoring` fails loudly at import time if the evaluation
# module has moved, rather than half way through a two-hour benchmark run.
_ensure_on_path(settings.EVALUATION_DIR)

from evaluate_results import summarize  # noqa: E402
from extractors import ClinicalTerms  # noqa: E402
from medical_metrics import METRICS_VERSION, evaluate  # noqa: E402
from semantic_metrics import available as semantic_available  # noqa: E402
from semantic_metrics import compute_batch as semantic_batch  # noqa: E402

__all__ = [
    "ClinicalTerms", "METRICS_VERSION", "evaluate", "summarize",
    "semantic_available", "semantic_batch",
    "model_registry", "build_stt_model", "torch_or_none",
]


# --- stt -------------------------------------------------------------------
# Imported lazily. `stt/app/config.py` imports torch at module level, and the
# dataset and scoring halves of this package have no business needing a
# multi-hundred-megabyte import to run their tests.
def _stt():
    _ensure_on_path(settings.STT_DIR)
    from app import config as stt_config
    from app import model as stt_model

    return stt_config, stt_model


def model_registry():
    """The models stt knows how to load, in stt's own preference order."""
    stt_config, _ = _stt()
    return dict(stt_config.MODEL_REGISTRY)


def build_stt_model(key, device):
    """Instantiate (without loading) one registered model, pinned to `device`.

    stt's own `build_model()` reads a single process-wide `config.DEVICE`, which
    is right for a service holding one model and wrong here: the whole point of
    two GPUs is two replicas on two different devices at once. Same registry,
    same classes, explicit device.
    """
    stt_config, stt_model = _stt()
    if key not in stt_config.MODEL_REGISTRY:
        raise KeyError(f"unknown model {key!r}; registered: {sorted(stt_config.MODEL_REGISTRY)}")
    spec = dict(stt_config.MODEL_REGISTRY[key])
    cls = stt_model._MODEL_TYPES[spec.pop("type")]
    return cls(model_id=spec.pop("model_id"), device=device, **spec)


def torch_or_none():
    """torch if it is installed, else None -- so CPU-only tests can skip."""
    try:
        import torch
    except ImportError:
        return None
    return torch
''')

In [ ]:
write("benchmark/dataset.py", r'''
"""What to benchmark: the recordings, and the ground truth to score them against.

A benchmark item is one recording plus, optionally, the radiologist-verified
text for it. The reference is optional on purpose -- ground truth arrives later
and more slowly than audio does, and a run over unlabelled recordings is still
useful (it produces the drafts that become labels). Unlabelled items are
transcribed and timed like any other; they are simply left out of the scores
rather than counted as failures.
"""
import json
import pathlib
from dataclasses import dataclass

import numpy as np
import soundfile as sf

import settings

AUDIO_EXTENSIONS = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".opus", ".webm", ".aac"}


@dataclass(frozen=True)
class Item:
    """One recording, and the text it should have produced."""
    asset_id: str
    audio: pathlib.Path
    reference: str | None = None

    @property
    def labelled(self):
        return bool(self.reference and self.reference.strip())


# --- Building a dataset ----------------------------------------------------
def from_json(manifest_path):
    """Read a manifest file.

    A JSON array of objects; `reference` may be inline text or the path to a
    `.txt` file, and both `audio` and `reference` paths are resolved relative to
    the manifest so a dataset directory can be moved as a unit.

        [{"asset_id": "DPM89130",
          "audio": "audio/DPM89130.mp3",
          "reference": "truth/DPM89130.txt"}]
    """
    manifest_path = pathlib.Path(manifest_path)
    base = manifest_path.parent
    entries = json.loads(manifest_path.read_text(encoding="utf-8"))

    items = []
    for entry in entries:
        audio = (base / entry["audio"]).resolve()
        items.append(Item(
            asset_id=entry.get("asset_id") or audio.stem,
            audio=audio,
            reference=_read_reference(entry.get("reference"), base),
        ))
    return items


def from_directory(audio_dir, reference_dir=None):
    """Pair `<audio_dir>/X.mp3` with `<reference_dir>/X.txt` by stem.

    The convention exists so a directory of recordings can be benchmarked with
    no manifest at all, and so labels can be dropped in one at a time as they
    are transcribed -- an audio file with no matching `.txt` is unlabelled, not
    an error.
    """
    audio_dir = pathlib.Path(audio_dir)
    reference_dir = pathlib.Path(reference_dir) if reference_dir else None

    items = []
    for path in sorted(audio_dir.iterdir()):
        if path.suffix.lower() not in AUDIO_EXTENSIONS:
            continue
        reference = None
        if reference_dir:
            truth = reference_dir / f"{path.stem}.txt"
            if truth.is_file():
                reference = truth.read_text(encoding="utf-8")
        items.append(Item(asset_id=path.stem, audio=path, reference=reference))
    return items


def _read_reference(value, base):
    """Inline text, or the contents of a `.txt` path relative to the manifest.

    Mirrors `evaluation/evaluate_results.py`, which accepts the same two forms,
    so a manifest written for one can be read by the other.
    """
    if not value:
        return None
    candidate = base / value
    if len(value) < 260 and candidate.is_file():
        return candidate.read_text(encoding="utf-8")
    return value


def describe(items):
    """A one-line census, printed before a run commits to hours of GPU time."""
    labelled = sum(1 for item in items if item.labelled)
    missing = [item.asset_id for item in items if not item.audio.is_file()]
    return {
        "items": len(items),
        "labelled": labelled,
        "unlabelled": len(items) - labelled,
        "missing_audio": missing,
    }


# --- Audio -----------------------------------------------------------------
def load_audio(path, target_sr=settings.TARGET_SAMPLE_RATE):
    """Decode to mono float32 at `target_sr`.

    Returned as one array rather than a path because every model in the
    registry takes `(audio, sr)`, and because decoding once per recording
    instead of once per model keeps the comparison about inference speed.
    """
    audio, sr = _decode(pathlib.Path(path))
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    audio = np.asarray(audio, dtype=np.float32)
    if sr != target_sr:
        audio, sr = _resample(audio, sr, target_sr), target_sr
    return audio, sr


def _decode(path):
    """soundfile first, ffmpeg second.

    libsndfile handles wav/flac/ogg and, since 1.1, mp3 -- but not m4a, and not
    every mp3 a phone produces. Rather than refuse those files, fall back to
    ffmpeg, which Kaggle and every Linux image already have.
    """
    try:
        return sf.read(str(path), dtype="float32", always_2d=False)
    except Exception as error:  # noqa: BLE001 - any decode failure is worth retrying
        decoded = _decode_via_ffmpeg(path)
        if decoded is None:
            raise RuntimeError(f"could not decode {path.name}: {error}") from error
        return decoded


def _decode_via_ffmpeg(path):
    import io
    import shutil
    import subprocess

    if shutil.which("ffmpeg") is None:
        return None
    result = subprocess.run(
        ["ffmpeg", "-nostdin", "-loglevel", "error", "-i", str(path),
         "-f", "wav", "-acodec", "pcm_s16le", "-"],
        capture_output=True, check=False)
    if result.returncode != 0 or not result.stdout:
        return None
    return sf.read(io.BytesIO(result.stdout), dtype="float32", always_2d=False)


def _resample(audio, sr, target_sr):
    """torchaudio's resampler when it is installed, linear interpolation when
    it is not -- a benchmark should not fail to start over a missing optional
    dependency, and every model receives the identical array either way."""
    try:
        import torch
        import torchaudio

        tensor = torch.from_numpy(audio).unsqueeze(0)
        return torchaudio.functional.resample(tensor, sr, target_sr).squeeze(0).numpy()
    except ImportError:
        count = int(round(len(audio) * target_sr / sr))
        source = np.linspace(0, len(audio) - 1, num=count, dtype=np.float64)
        return np.interp(source, np.arange(len(audio)), audio).astype(np.float32)
''')

In [ ]:
write("benchmark/transcribe.py", r'''
"""Running the models: windowing, stitching, and one replica per GPU.

Three things here exist to keep a comparison honest rather than to make it
fast:

* **Windowing.** Whisper's encoder is fixed at 30 seconds. Left alone it
  transcribes the first half-minute of a four-minute dictation and returns,
  which reads as a catastrophic WER caused by the model rather than by the
  harness. Every model gets the same overlapping windows.
* **One replica per device.** Both T4s run the same model over different halves
  of the batch, so throughput doubles without changing a single number in the
  result -- the alternative, one model per GPU, would have the two models
  competing for bandwidth and make the latency figures meaningless.
* **Failures are recorded, not raised.** One model that OOMs on one recording
  must not discard the other nine models' results.
"""
import threading
import time
from dataclasses import asdict, dataclass, field

import bridge
import dataset
import settings


@dataclass
class Transcript:
    """One model's attempt at one recording."""
    asset_id: str
    model: str
    text: str = ""
    audio_seconds: float = 0.0
    elapsed_seconds: float = 0.0
    windows: int = 0
    device: str = ""
    error: str | None = None

    @property
    def real_time_factor(self):
        """Seconds of compute per second of audio. Below 1.0 is faster than
        real time, which is the bar for dictating and reading back in a clinic."""
        return self.elapsed_seconds / self.audio_seconds if self.audio_seconds else 0.0

    def as_dict(self):
        return {**asdict(self), "real_time_factor": round(self.real_time_factor, 4)}


@dataclass
class ModelRun:
    """Everything one model produced over the whole batch."""
    model: str
    model_id: str = ""
    devices: list = field(default_factory=list)
    transcripts: list = field(default_factory=list)
    load_seconds: float = 0.0
    peak_vram_bytes: int = 0

    def summary(self):
        done = [t for t in self.transcripts if t.error is None]
        audio = sum(t.audio_seconds for t in done)
        compute = sum(t.elapsed_seconds for t in done)
        return {
            "model": self.model,
            "model_id": self.model_id,
            "devices": self.devices,
            "transcribed": len(done),
            "failed": len(self.transcripts) - len(done),
            "load_seconds": round(self.load_seconds, 2),
            "peak_vram_gb": round(self.peak_vram_bytes / 1024 ** 3, 2),
            "audio_seconds": round(audio, 1),
            # Aggregate RTF from summed totals, not a mean of per-file ratios:
            # a two-second clip with a fixed startup cost would otherwise
            # dominate an average built from hours of real dictation.
            "real_time_factor": round(compute / audio, 4) if audio else 0.0,
        }


# --- Windowing -------------------------------------------------------------
def plan_windows(duration, window_sec=None, overlap_sec=None):
    """Cut `duration` into overlapping windows, as [(start, end)] in seconds.

    The overlap is what makes stitching possible: a word landing on a cut is
    spoken fully inside one of the two neighbours.
    """
    window_sec = settings.WINDOW_SEC if window_sec is None else window_sec
    overlap_sec = settings.OVERLAP_SEC if overlap_sec is None else overlap_sec
    if duration <= window_sec:
        return [(0.0, duration)]
    if overlap_sec >= window_sec:
        raise ValueError("overlap must be shorter than the window")

    step = window_sec - overlap_sec
    windows = []
    start = 0.0
    while start < duration:
        end = min(start + window_sec, duration)
        windows.append((start, end))
        if end >= duration:
            break
        start += step
    return windows


def stitch(parts, max_overlap_words=None):
    """Join per-window transcripts, dropping what the overlap said twice.

    Where one window ends with the same words the next begins with, that is the
    shared audio being transcribed once each. The longest such run is the seam;
    no match means the two windows disagreed about the overlap, in which case
    keeping both is the safer error -- an omission is invisible to a reader, a
    duplication is not.
    """
    cap = settings.MAX_STITCH_OVERLAP_WORDS if max_overlap_words is None else max_overlap_words
    merged = []
    for part in parts:
        words = part.split()
        if not words:
            continue
        if not merged:
            merged = words
            continue
        merged.extend(words[_seam(merged, words, cap):])
    return " ".join(merged)


def _seam(left, right, cap):
    """Length of the longest suffix of `left` that is also a prefix of `right`."""
    limit = min(cap, len(left), len(right))
    for length in range(limit, 0, -1):
        if left[-length:] == right[:length]:
            return length
    return 0


# --- Running one model over a batch ----------------------------------------
def transcribe_batch(model_key, items, devices=None, language=None,
                     model_factory=None, on_progress=None, **window_kwargs):
    """Load `model_key` on every device and transcribe `items` across them.

    `model_factory(key, device)` is injected so the tests can run the whole
    scheduling, windowing and stitching path against a stub, with no weights
    and no GPU.
    """
    devices = list(devices or resolve_devices())
    factory = model_factory or bridge.build_stt_model
    language = language or settings.DEFAULT_LANGUAGE

    run = ModelRun(model=model_key, devices=devices)
    shards = [items[index::len(devices)] for index in range(len(devices))]
    lock = threading.Lock()

    _reset_vram(devices)
    threads = [
        threading.Thread(
            target=_work_shard, name=f"{model_key}@{device}",
            args=(factory, model_key, device, shard, language, run, lock,
                  on_progress, window_kwargs))
        for device, shard in zip(devices, shards) if shard
    ]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()

    run.peak_vram_bytes = _peak_vram(devices)
    # Ordered as the caller supplied them; the shards finish interleaved.
    order = {item.asset_id: index for index, item in enumerate(items)}
    run.transcripts.sort(key=lambda t: order.get(t.asset_id, 0))
    run.load_seconds = round(run.load_seconds, 2)
    return run


def _work_shard(factory, model_key, device, shard, language, run, lock,
                on_progress, window_kwargs):
    """One device's share of the batch: load once, then transcribe in turn."""
    try:
        loading = time.perf_counter()
        model = factory(model_key, device)
        model.load()
        load_seconds = time.perf_counter() - loading
    except Exception as error:  # noqa: BLE001 - a model that will not load is a result
        with lock:
            run.transcripts.extend(
                Transcript(asset_id=item.asset_id, model=model_key, device=device,
                           error=f"load failed: {error}")
                for item in shard)
        return

    with lock:
        run.load_seconds = max(run.load_seconds, load_seconds)
        run.model_id = getattr(model, "model_id", "")

    try:
        for item in shard:
            transcript = _transcribe_one(model, model_key, device, item, language, window_kwargs)
            with lock:
                run.transcripts.append(transcript)
            if on_progress:
                on_progress(transcript)
    finally:
        # Free the replica before the next model loads; two large models
        # resident at once is how a 16 GB T4 runs out.
        model.unload()


def _transcribe_one(model, model_key, device, item, language, window_kwargs):
    transcript = Transcript(asset_id=item.asset_id, model=model_key, device=device)
    try:
        audio, sr = dataset.load_audio(item.audio)
        transcript.audio_seconds = len(audio) / sr
        windows = plan_windows(transcript.audio_seconds, **window_kwargs)
        transcript.windows = len(windows)

        started = time.perf_counter()
        parts = [
            model.transcribe(audio[int(start * sr):int(end * sr)], sr, language=language)
            for start, end in windows
        ]
        transcript.elapsed_seconds = time.perf_counter() - started
        transcript.text = stitch(parts)
    except Exception as error:  # noqa: BLE001 - recorded per recording, never fatal
        transcript.error = f"{type(error).__name__}: {error}"
    return transcript


class EchoModel:
    """A stand-in that loads nothing and transcribes nothing.

    A benchmark run downloads gigabytes of weights and then spends hours on
    them, which is a long way to travel before finding out a path was wrong.
    Swapping this in exercises the whole harness -- decoding, windowing,
    stitching, scoring, writing -- in seconds, so the only thing left to be
    wrong is the models themselves.
    """

    def __init__(self, key="dry-run", device="cpu"):
        self.model_id = f"dry-run/{key}"
        self.device = device

    def load(self):
        pass

    def unload(self):
        pass

    def transcribe(self, audio, sr, language=None):
        return ""


def dry_run_factory(key, device):
    return EchoModel(key, device)


# --- Devices ---------------------------------------------------------------
def resolve_devices(spec=None):
    """Turn the `DEVICES` setting into a concrete list of torch devices."""
    spec = settings.DEVICES if spec is None else spec
    if spec and spec != "auto":
        return [device.strip() for device in spec.split(",") if device.strip()]

    torch = bridge.torch_or_none()
    if torch is None or not torch.cuda.is_available():
        return ["cpu"]
    return [f"cuda:{index}" for index in range(torch.cuda.device_count())]


def describe_devices(devices=None):
    """What the run is about to use -- printed so a Kaggle notebook that
    quietly fell back to CPU says so before spending an hour proving it."""
    devices = devices or resolve_devices()
    torch = bridge.torch_or_none()
    described = []
    for device in devices:
        if torch is not None and device.startswith("cuda"):
            index = int(device.split(":")[1]) if ":" in device else 0
            properties = torch.cuda.get_device_properties(index)
            described.append({"device": device, "name": properties.name,
                              "total_vram_gb": round(properties.total_memory / 1024 ** 3, 1)})
        else:
            described.append({"device": device, "name": "cpu", "total_vram_gb": 0.0})
    return described


def _reset_vram(devices):
    torch = bridge.torch_or_none()
    if torch is None:
        return
    for device in devices:
        if device.startswith("cuda"):
            torch.cuda.reset_peak_memory_stats(device)


def _peak_vram(devices):
    """Peak across devices, not the sum: the replicas are identical, so the
    number to report is what one GPU had to hold."""
    torch = bridge.torch_or_none()
    if torch is None:
        return 0
    peaks = [torch.cuda.max_memory_allocated(device)
             for device in devices if device.startswith("cuda")]
    return max(peaks, default=0)
''')

In [ ]:
write("benchmark/scoring.py", r'''
"""Scoring the transcripts -- by calling `evaluation/`, never by reimplementing it.

Every number here comes from `medical_metrics.evaluate()` and
`evaluate_results.summarize()`, the same two functions the live service and the
batch CLI use. That is the point: a benchmark that computed its own WER would
eventually disagree with production about which model is better, and there
would be no way to tell which of the two was wrong.

What this module adds is the join -- lining transcripts up with their
references, keeping the unlabelled ones out of the arithmetic, and carrying the
speed measurements through to the summary, since a model is only usable if it
is both accurate and fast enough.
"""
import bridge

# Where a long dictation stops behaving like a short one. Radiology audio spans
# a one-line finding to a multi-minute study, and a model that is fine on the
# first and collapses on the second has an average that says neither.
DURATION_BUCKETS = [("lt_30s", 0, 30), ("30s_2m", 30, 120), ("gt_2m", 120, float("inf"))]


def score_run(run, items, terms):
    """Score one model's transcripts against the labelled items.

    Returns result dicts in exactly the shape `evaluate_results.summarize()`
    consumes, so the aggregate step is the module's own. Semantic metrics are
    deliberately not computed here -- see `attach_semantic()`.
    """
    references = {item.asset_id: item.reference for item in items if item.labelled}

    results = []
    for transcript in run.transcripts:
        reference = references.get(transcript.asset_id)
        if reference is None:
            continue  # unlabelled: transcribed and timed, but nothing to score against
        if transcript.error is not None:
            # A failed transcription is an empty output, not a missing data
            # point. Dropping it would flatter the model that crashed.
            hypothesis = ""
        else:
            hypothesis = transcript.text

        results.append({
            "asset_id": transcript.asset_id,
            "model": run.model,
            "model_version": run.model_id,
            "pipeline": "stt_only",
            "transcription_error": transcript.error,
            "real_time_factor": round(transcript.real_time_factor, 4),
            "audio_seconds": round(transcript.audio_seconds, 2),
            "reference": reference,
            "hypothesis": hypothesis,
            **bridge.evaluate(hypothesis, reference, terms),
        })
    return results


def attach_semantic(results):
    """Add the two embedding metrics to every result, in one pass.

    Kept out of `score_run` on purpose. `evaluate(include_semantic=True)` scores
    one report at a time, which for a benchmark means a batch of one, several
    hundred times over, against models that take longer to load than to run.
    Here every pair goes through together.
    """
    if not results:
        return results

    scores = bridge.semantic_batch([(r["reference"], r["hypothesis"]) for r in results])
    for result, semantic in zip(results, scores):
        result["semantic"] = semantic
    return results


def add_semantic(results, summary):
    """Compute the embedding metrics after the fact and fold them into both
    levels -- per report, and as a per-model mean on the summary.

    Exists so the decision to spend the extra model load can be made after
    seeing the WER, rather than committed to before the run starts.
    """
    attach_semantic(results)
    for bucket in summary.get("models", []):
        bucket["semantic"] = _mean_semantic(
            [r for r in results if r["model"] == bucket["model"]])
    return results, summary


def duration_buckets(results):
    """WER by recording length -- summed edits over summed words, per bucket.

    Long audio is where windowing and stitching can go wrong, and where a model
    with a short attention span quietly degrades. One overall WER hides both.
    """
    summary = {}
    for name, low, high in DURATION_BUCKETS:
        inside = [r for r in results if low <= r.get("audio_seconds", 0) < high]
        if not inside:
            continue
        errors = sum(r["general"]["substitutions"] + r["general"]["insertions"]
                     + r["general"]["deletions"] for r in inside)
        words = sum(r["general"]["reference_words"] for r in inside)
        summary[name] = {
            "reports": len(inside),
            "wer": round(errors / words, 4) if words else 0.0,
            "audio_seconds": round(sum(r["audio_seconds"] for r in inside), 1),
        }
    return summary


def score_all(runs, items, terms=None, include_semantic=False):
    """Score every model's run and aggregate them into one comparison.

    The summary is `evaluate_results.summarize()` output with the speed figures
    folded in per model, so accuracy and cost are read off the same row.
    """
    terms = terms or bridge.ClinicalTerms()

    results = []
    speed = {}
    for run in runs:
        results.extend(score_run(run, items, terms))
        speed[run.model] = run.summary()

    if include_semantic:
        attach_semantic(results)

    summary = bridge.summarize(results, terms) if results else {
        "models": [], "reports": 0,
        "evaluation": {"metrics_version": bridge.METRICS_VERSION,
                       "terms_version": terms.version, "terms_sha": terms.sha},
    }
    for bucket in summary["models"]:
        model_results = [r for r in results if r["model"] == bucket["model"]]
        bucket["speed"] = speed.get(bucket["model"], {})
        bucket["by_duration"] = duration_buckets(model_results)
        if include_semantic:
            bucket["semantic"] = _mean_semantic(model_results)

    # Models whose every recording was unlabelled never reach summarize(),
    # because there was nothing to score. Report their speed anyway rather than
    # letting them vanish from the run.
    scored = {bucket["model"] for bucket in summary["models"]}
    summary["unscored_models"] = [
        {"model": model, "speed": figures, "reason": "no labelled recordings"}
        for model, figures in speed.items() if model not in scored
    ]
    summary["labelled"] = sum(1 for item in items if item.labelled)
    summary["unlabelled"] = sum(1 for item in items if not item.labelled)
    return results, summary


def _mean_semantic(results):
    """Both embedding scores are similarities in [0, 1], so a plain mean is the
    right aggregate -- unlike an error rate, there is no denominator to sum."""
    scored = [r["semantic"] for r in results if r.get("semantic")]
    if not scored:
        return {}
    return {f"{field}_mean": round(sum(s[field] for s in scored) / len(scored), 4)
            for field in ("bertscore_f1", "semantic_similarity")}
''')

In [ ]:
write("benchmark/leaderboard.py", r'''
"""Turning a summary into something a person can read.

The leaderboard deliberately shows accuracy and cost side by side. A model that
wins on WER while running at four times real time on a T4 has not won anything
a clinic can deploy, and a table that hides the second number invites exactly
that mistake.
"""
import csv
import json
import pathlib

# Column key, heading, format. Order is the reading order.
COLUMNS = [
    ("model", "model", "{}"),
    ("reports", "n", "{}"),
    # The headline is the corpus WER -- total edits over total words -- not the
    # median of per-report WERs. The percentiles beside it describe the spread.
    ("corpus_wer", "WER", "{:.3f}"),
    ("corpus_cer", "CER", "{:.3f}"),
    ("wer_p50", "p50", "{:.3f}"),
    ("wer_p90", "p90", "{:.3f}"),
    ("chrf_mean", "chrF", "{:.3f}"),
    ("medical_term_f1", "term F1", "{:.3f}"),
    # Rates, not raw counts: two models that saw the same references have the
    # same denominators, but the rate is what survives a change of corpus.
    ("negation_error_rate", "neg err", "{:.1%}"),
    ("laterality_error_rate", "lat err", "{:.1%}"),
    ("number_error_rate", "num err", "{:.1%}"),
    ("unit_error_rate", "unit err", "{:.1%}"),
    ("critical_omission_rate", "crit om", "{:.1%}"),
    ("unsupported_addition_rate", "unsup add", "{:.1%}"),
    ("repetition_score_p95", "rep p95", "{:.2f}"),
    # Shown next to the reference's own figure, never alone: this corpus
    # code-switches English terms on purpose, so the gap is the signal.
    ("script_contamination_mean", "latin", "{:.1%}"),
    ("reference_script_contamination_mean", "latin(ref)", "{:.1%}"),
    ("insertion_rate", "ins", "{:.3f}"),
    ("review_rate", "review", "{:.0%}"),
    ("pct_catastrophic", "catastr", "{:.0%}"),
    ("real_time_factor", "RTF", "{:.2f}"),
    ("throughput", "xRT", "{:.1f}"),
    ("peak_vram_gb", "VRAM GB", "{:.1f}"),
]

# Default ranking. Corpus WER is the number the field reports, so it is the
# number the table sorts on.
SORT_BY = "corpus_wer"


def rows(summary, sort_by=None):
    """One flat row per model, best first.

    Flattening matters: `summarize()` nests counts, rates, reliability and
    speed in four places, and a comparison table is unreadable if the reader
    has to remember which number lives where.
    """
    sort_by = sort_by or SORT_BY
    flattened = []
    for bucket in summary.get("models", []):
        reliability = bucket.get("reliability", {})
        flattened.append({
            "model": bucket["model"],
            "reports": bucket["reports"],
            **{key: reliability.get(key, 0.0) for key in
               ("wer_p50", "wer_p90", "wer_p95", "pct_perfect", "pct_catastrophic",
                "ser", "empty_output_rate")},
            **bucket.get("corpus", {}),
            **bucket.get("text", {}),
            **bucket.get("semantic", {}),
            **bucket.get("rates", {}),
            **{key: bucket.get(key, 0) for key in
               ("negation_errors", "laterality_errors", "number_errors", "unit_errors",
                "critical_omissions", "unsupported_additions",
                "negation_scored", "laterality_scored", "reference_measurements",
                "critical_omission_scored", "unsupported_addition_scored")},
            "real_time_factor": bucket.get("speed", {}).get("real_time_factor", 0.0),
            "peak_vram_gb": bucket.get("speed", {}).get("peak_vram_gb", 0.0),
            "load_seconds": bucket.get("speed", {}).get("load_seconds", 0.0),
            "failed": bucket.get("speed", {}).get("failed", 0),
            # Hours of audio per hour of wall clock -- the deployment question
            # RTF answers backwards.
            "throughput": round(1.0 / rtf, 2) if (rtf := bucket.get("speed", {}).get(
                "real_time_factor", 0.0)) else 0.0,
            **{f"wer_{name}": figures["wer"]
               for name, figures in bucket.get("by_duration", {}).items()},
        })
    return sorted(flattened, key=lambda row: row.get(sort_by, 0))


def to_markdown(summary, sort_by=None):
    """A markdown table -- readable in a terminal, rendered in a notebook."""
    table = rows(summary, sort_by)
    if not table:
        return "_no scored models_"

    header = "| " + " | ".join(heading for _, heading, _ in COLUMNS) + " |"
    rule = "|" + "|".join("---" for _ in COLUMNS) + "|"
    lines = [header, rule]
    for row in table:
        lines.append("| " + " | ".join(
            _cell(row.get(key), template) for key, _, template in COLUMNS) + " |")
    return "\n".join(lines)


def _cell(value, template):
    if value is None:
        return "-"
    try:
        return template.format(value)
    except (TypeError, ValueError):
        return str(value)


def worst(results, count=10, by="wer"):
    """The recordings that went worst, so a run ends with something to look at.

    An aggregate says a model is 12% wrong; it never says which twelve percent.
    These are the rows to read before trusting any of the other numbers.
    """
    ranked = sorted(results, key=lambda result: -result["general"].get(by, 0))
    return [{
        "asset_id": result["asset_id"],
        "model": result["model"],
        by: result["general"].get(by),
        "repetition_score": result["general"].get("repetition_score"),
        "hallucination_ratio": result["general"].get("hallucination_ratio"),
        "review_reasons": result.get("review_reasons", []),
        "critical_errors": len(result.get("critical_errors", [])),
    } for result in ranked[:count]]


def to_dataframe(summary, sort_by=None):
    """The same rows as a pandas DataFrame, for sorting and plotting in a
    notebook. pandas is not a dependency of the benchmark itself, so this is
    only importable where it is already installed."""
    import pandas as pd

    return pd.DataFrame(rows(summary, sort_by))


def check_writable(out_dir):
    """Fail now rather than after the GPU time is spent.

    The results are written at the end of a run that can take hours. A path
    that cannot be created -- the usual case being a Kaggle notebook pointed at
    the read-only `/kaggle/input` mount -- should say so in the first second,
    not the last.
    """
    out_dir = pathlib.Path(out_dir)
    try:
        out_dir.mkdir(parents=True, exist_ok=True)
        probe = out_dir / ".write-check"
        probe.write_text("", encoding="utf-8")
        probe.unlink()
    except OSError as error:
        raise OSError(
            f"cannot write results to {out_dir}: {error}. "
            f"On Kaggle use a path under /kaggle/working.") from error
    return out_dir


def per_report_rows(results):
    """One flat row per recording per model, for a spreadsheet.

    The nested JSON is the record of what happened; this is the form a person
    actually reads a few hundred rows in. Reference and hypothesis travel with
    the numbers so a bad score can be looked at rather than just counted.
    """
    flattened = []
    for result in results:
        flattened.append({
            "asset_id": result["asset_id"],
            "model": result["model"],
            "audio_seconds": result.get("audio_seconds"),
            "real_time_factor": result.get("real_time_factor"),
            **result["general"],
            **result["clinical_metrics"],
            **{f"n_{key}": value for key, value in result["clinical_counts"].items()},
            **(result.get("semantic") or {}),
            "requires_medical_review": result["requires_medical_review"],
            "review_reasons": ";".join(result.get("review_reasons", [])),
            "critical_errors": len(result.get("critical_errors", [])),
            "transcription_error": result.get("transcription_error") or "",
            "reference": result.get("reference", ""),
            "hypothesis": result.get("hypothesis", ""),
        })
    return flattened


def _write_csv(path, table):
    if not table:
        return
    # utf-8-sig: Excel reads Persian text as mojibake without the BOM.
    columns = list(dict.fromkeys(key for row in table for key in row))
    with path.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(table)


def write(out_dir, results, summary, runs=None):
    """Persist a run: per-report scores, the summary, and the raw transcripts.

    Written as both JSON and CSV -- JSON keeps the nesting and is what gets
    re-read, CSV is what gets opened and sorted by a person.

    Transcripts are written separately and always, including for unlabelled
    recordings -- they are the input to the next labelling round, not a
    by-product of this one.
    """
    out_dir = pathlib.Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    _dump(out_dir / "summary.json", summary)
    _dump(out_dir / "results.json", results)
    _dump(out_dir / "leaderboard.md", to_markdown(summary), raw=True)
    _write_csv(out_dir / "leaderboard.csv", rows(summary))
    _write_csv(out_dir / "per_report.csv", per_report_rows(results))

    if runs:
        transcripts = {}
        for run in runs:
            transcripts[run.model] = {
                "summary": run.summary(),
                "transcripts": [t.as_dict() for t in run.transcripts],
            }
        _dump(out_dir / "transcripts.json", transcripts)
    return out_dir


def _dump(path, payload, raw=False):
    text = payload if raw else json.dumps(payload, ensure_ascii=False, indent=2)
    path.write_text(text, encoding="utf-8")
''')

In [ ]:
write("benchmark/run_benchmark.py", r'''
"""Run the benchmark: audio in, a ranked comparison out.

    python run_benchmark.py --audio recordings/ --truth labels/ --models whisper,seamless
    python run_benchmark.py --manifest dataset/manifest.json --out results/run-01

Models are loaded one at a time and replicated across every available GPU, so
peak memory is one model's worth however many are being compared. The notebook
in `notebooks/` calls `run()` directly with the same arguments.
"""
import argparse
import sys

import bridge
import dataset
import leaderboard
import scoring
import settings
import transcribe


def run(items, models=None, devices=None, language=None, terms_path=None,
        include_semantic=False, on_progress=None, model_factory=None, **window_kwargs):
    """Transcribe `items` with each model, then score everything at once.

    Scoring happens after all transcription rather than per model, because
    `summarize()` compares models against each other and needs them together.

    `model_factory` overrides how models are built -- pass
    `transcribe.dry_run_factory` to exercise the harness without weights.
    """
    models = list(models or settings.DEFAULT_MODELS)
    devices = list(devices or transcribe.resolve_devices())
    terms = bridge.ClinicalTerms(terms_path)

    runs = []
    for key in models:
        runs.append(transcribe.transcribe_batch(
            key, items, devices=devices, language=language,
            model_factory=model_factory, on_progress=on_progress, **window_kwargs))

    results, summary = scoring.score_all(runs, items, terms, include_semantic)
    summary["devices"] = transcribe.describe_devices(devices)
    summary["windowing"] = {
        "window_sec": window_kwargs.get("window_sec", settings.WINDOW_SEC),
        "overlap_sec": window_kwargs.get("overlap_sec", settings.OVERLAP_SEC),
    }
    return runs, results, summary


def load_items(args):
    """Whichever way the caller described the dataset."""
    if args.manifest:
        return dataset.from_json(args.manifest)
    if args.audio:
        return dataset.from_directory(args.audio, args.truth)
    raise SystemExit("give either --manifest or --audio")


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    source = parser.add_argument_group("dataset")
    source.add_argument("--manifest", help="JSON array of {asset_id, audio, reference}")
    source.add_argument("--audio", help="directory of recordings")
    source.add_argument("--truth", help="directory of <stem>.txt ground truth files")

    parser.add_argument("--models", default=None,
                        help="comma-separated stt registry keys (default: settings.DEFAULT_MODELS)")
    parser.add_argument("--devices", default=None,
                        help="comma-separated torch devices, e.g. cuda:0,cuda:1 (default: all GPUs)")
    parser.add_argument("--language", default=None, help="language code passed to each model")
    parser.add_argument("--out", default=str(settings.OUT_DIR), help="output directory")
    parser.add_argument("--terms", default=None, help="path to clinical_terms.json")
    parser.add_argument("--window-sec", type=float, default=settings.WINDOW_SEC)
    parser.add_argument("--overlap-sec", type=float, default=settings.OVERLAP_SEC)
    parser.add_argument("--semantic", action="store_true",
                        help="also compute BERTScore and semantic similarity (loads a model)")
    parser.add_argument("--list-models", action="store_true", help="print the registry and exit")
    parser.add_argument("--dry-run", action="store_true",
                        help="exercise decoding, windowing and scoring with a stub model, "
                             "so a wrong path costs seconds instead of a weights download")
    args = parser.parse_args(argv)

    if args.list_models:
        for key, spec in bridge.model_registry().items():
            print(f"  {key:24} {spec['model_id']}")
        return 0

    # Everything that can be checked cheaply is checked before any weights load:
    # a run that fails on the last line has already spent the GPU time.
    try:
        out_dir = leaderboard.check_writable(args.out)
    except OSError as error:
        raise SystemExit(str(error)) from None

    items = load_items(args)
    census = dataset.describe(items)
    if census["missing_audio"]:
        raise SystemExit(f"missing audio for: {', '.join(census['missing_audio'][:5])}")
    if not items:
        raise SystemExit("no recordings found")

    models = ([key.strip() for key in args.models.split(",") if key.strip()]
              if args.models else None)
    devices = transcribe.resolve_devices(args.devices)
    if args.dry_run:
        models = models or ["dry-run"]
        print("DRY RUN: stub model, no weights loaded -- scores are meaningless")

    print(f"{census['items']} recording(s): "
          f"{census['labelled']} labelled, {census['unlabelled']} unlabelled")
    for device in transcribe.describe_devices(devices):
        print(f"  {device['device']:9} {device['name']} ({device['total_vram_gb']} GB)")

    runs, results, summary = run(
        items, models=models, devices=devices, language=args.language,
        terms_path=args.terms, include_semantic=args.semantic,
        model_factory=transcribe.dry_run_factory if args.dry_run else None,
        on_progress=_progress, window_sec=args.window_sec, overlap_sec=args.overlap_sec)

    leaderboard.write(out_dir, results, summary, runs)

    print()
    print(leaderboard.to_markdown(summary))
    if summary["unscored_models"]:
        names = ", ".join(entry["model"] for entry in summary["unscored_models"])
        print(f"\nnot scored (no labelled recordings): {names}")
    print(f"\nwritten to {out_dir}")
    return 0


def _progress(transcript):
    state = transcript.error or f"{transcript.real_time_factor:.2f}x real time"
    print(f"  [{transcript.model}] {transcript.asset_id}: {state}", flush=True)


if __name__ == "__main__":
    sys.exit(main())
''')

In [ ]:
import bridge, dataset, leaderboard, run_benchmark, scoring, transcribe

print("modules loaded")
print("metrics version:", bridge.METRICS_VERSION)
print("models available:")
for key, spec in bridge.model_registry().items():
    print(f"   {key:24} {spec['model_id']}")

## 3. Configure the run

`AUDIO_DIR` holds the recordings. `TRUTH_DIR` holds `<same stem>.txt` ground
truth files — leave it `None` for a first pass over unlabelled audio, which
still produces transcripts and timings.

`MODELS` are keys from the registry printed above.

In [ ]:
AUDIO_DIR = "/kaggle/input/buali-audio/audio"     # <- your recordings
TRUTH_DIR = "/kaggle/input/buali-audio/truth"     # <- or None if unlabelled
MODELS    = ["whisper", "whisper-large-v3-turbo", "seamless"]
LANGUAGE  = "fa"

# /kaggle/working is the writable half of the runtime, and the only directory
# Kaggle keeps when the session ends.
OUT_DIR   = pathlib.Path("/kaggle/working/benchmark_results")

# Whisper's encoder is fixed at 30 seconds; every model gets the same windows so
# a long-audio penalty is never mistaken for a model being worse.
WINDOW_SEC, OVERLAP_SEC = 28.0, 3.0

## 4. Load the dataset

In [ ]:
items = dataset.from_directory(AUDIO_DIR, TRUTH_DIR)
census = dataset.describe(items)
census

In [ ]:
assert items, "no audio found — check AUDIO_DIR"
assert not census["missing_audio"], census["missing_audio"][:5]

# Prove the results can be written before spending the GPU time, not after.
leaderboard.check_writable(OUT_DIR)

for item in items[:5]:
    audio, sr = dataset.load_audio(item.audio)
    print(f"  {item.asset_id:20} {len(audio)/sr:7.1f}s  labelled={item.labelled}")

### Dry run first

A stub model, no weights. Exercises decoding, windowing, stitching, scoring and
writing in seconds — so a wrong path costs you that, and not a 10 GB download
followed by a failure.

In [ ]:
_, dry_results, _ = run_benchmark.run(
    items[:3], models=["dry-run"], model_factory=transcribe.dry_run_factory,
    window_sec=WINDOW_SEC, overlap_sec=OVERLAP_SEC)
print(f"plumbing OK — {len(dry_results)} labelled recording(s) scored end to end")

## 5. Run

One model at a time, replicated across both GPUs with the batch split between
them: throughput doubles and no number changes. Peak memory stays one model's
worth however many are being compared.

Roughly *(audio hours) × (models) × RTF*. Progress prints per recording.

In [ ]:
for device in transcribe.describe_devices():
    print(f"  {device['device']:9} {device['name']} ({device['total_vram_gb']} GB)")

runs, results, summary = run_benchmark.run(
    items,
    models=MODELS,
    language=LANGUAGE,
    window_sec=WINDOW_SEC,
    overlap_sec=OVERLAP_SEC,
    on_progress=lambda t: print(f"  [{t.model}] {t.asset_id}: "
                                f"{t.error or f'{t.real_time_factor:.2f}x real time'}",
                                flush=True),
)
print("\ndone")

## 6. Optional — the two embedding metrics

BERTScore and semantic similarity load a second model, so they are off by
default. This adds them for the whole batch in one pass; skip it if you only
need WER and the clinical metrics.

In [ ]:
# !pip install -q bert-score sentence-transformers   # uncomment on first run

usable, reason = bridge.semantic_available()
print("semantic extras:", "available" if usable else f"not installed — {reason}")

if usable:
    scoring.add_semantic(results, summary)
    print("added:", list(summary["models"][0]["semantic"]))

## 7. The batch leaderboard

One row per model over the whole batch, accuracy and cost side by side on
purpose: a model that wins on WER while running at four times real time on a T4
has not won anything deployable.

**`WER` is the corpus WER** — total edits over total reference words — not the
mean of the per-report WERs, which would weight a one-line finding exactly like
a multi-minute study. `p50`/`p90` beside it describe the spread.

`latin` / `latin(ref)` are script contamination: the share of letters that are
not Arabic-script, in the transcript and in the ground truth. **Read the gap,
not the number** — this corpus code-switches English radiology terms on purpose,
so a correct transcript is "contaminated" too.

In [ ]:
import pandas as pd

board = leaderboard.to_dataframe(summary)
board[["model", "reports", "corpus_wer", "corpus_cer", "wer_p50", "wer_p90",
       "chrf_mean", "medical_term_f1", "negation_error_rate",
       "laterality_error_rate", "number_error_rate", "unit_error_rate",
       "critical_omission_rate", "unsupported_addition_rate",
       "repetition_score_p95", "script_contamination_mean",
       "reference_script_contamination_mean", "insertion_rate", "review_rate",
       "pct_catastrophic", "real_time_factor", "throughput", "peak_vram_gb"]]

In [ ]:
from IPython.display import Markdown

Markdown(leaderboard.to_markdown(summary))

### WER by recording length

Long audio is where windowing and stitching can go wrong, and where a model with
a short attention span quietly degrades. One overall WER hides both.

In [ ]:
pd.DataFrame({
    bucket["model"]: {name: figures["wer"]
                      for name, figures in bucket["by_duration"].items()}
    for bucket in summary["models"]
}).T

In [ ]:
# Everything summarize() produced for one model. The table above is a readable
# subset; this is the full set.
import json

print(json.dumps(summary["models"][0], ensure_ascii=False, indent=2)[:3000])

## 8. What went wrong

An aggregate says a model is 12% wrong; it never says *which* 12%. These are the
recordings to read before trusting any of the numbers above.

In [ ]:
pd.DataFrame(leaderboard.worst(results, count=15))

In [ ]:
from collections import Counter

flagged = [r for r in results if r["requires_medical_review"]]
print(f"{len(flagged)} of {len(results)} scored reports need review\n")
Counter(reason for r in flagged for reason in r["review_reasons"]).most_common()

## 9. Read one recording side by side

Set `ASSET` to anything from the table above. Critical errors are the ones that
change what a report means — a flipped negation, a wrong side, a wrong number.

In [ ]:
assert results, "nothing was scored — this run had no ground truth"
ASSET = results[0]["asset_id"]

reference = next(i.reference for i in items if i.asset_id == ASSET)
print(f"REFERENCE\n{reference}\n")

for result in [r for r in results if r["asset_id"] == ASSET]:
    said = next(t.text for run in runs for t in run.transcripts
                if t.asset_id == ASSET and t.model == result["model"])
    print(f"--- {result['model']}  WER={result['general']['wer']:.3f} "
          f"F1={result['clinical_metrics']['medical_term_f1']:.3f}")
    print(said)
    for error in result["critical_errors"]:
        print(f"    ! {error['type']}: {error['reference']} -> {error['prediction']}")
    print()

## 10. Save

In [ ]:
leaderboard.write(OUT_DIR, results, summary, runs)

for path in sorted(OUT_DIR.iterdir()):
    print(f"  {path.name:20} {path.stat().st_size / 1024:9.1f} KB")

| File | What it is |
|---|---|
| `leaderboard.csv` | one row per model — the table above |
| `leaderboard.md` | the same, as markdown |
| `per_report.csv` | one row per recording per model, with reference and hypothesis |
| `summary.json` | the full nested batch summary |
| `results.json` | the full per-report scores |
| `transcripts.json` | what every model said, including unlabelled recordings |

Both CSVs are UTF-8 with a BOM, so Excel reads the Persian correctly.

`transcripts.json` covers recordings with no ground truth too — those drafts are
the starting point for the next labelling round.